# Project HOLLYWOOD — Experimental Pipeline

**Pipeline:** IDF weighting → UMAP → HDBSCAN → **XGBoost Validation, Interpretation & Outlier Recovery**

This notebook builds a content-based movie recommendation system that groups ~8,000 films into streaming-service-style "rails" (think Netflix categories like *Gritty Crime Dramas* or *Lighthearted Family Adventures*). Instead of relying on simple genre tags, it uses **248 continuous genome scores** per movie — nuanced measurements of things like *revenge*, *claymation*, *thought-provoking*, *stylized violence* — to find groups of movies that genuinely *feel* alike.

The pipeline works in stages:

1. **IDF Weighting** — Reweight the 248 genome features so rare, distinctive tags matter more than ubiquitous ones. A tag like "claymation" (present in 45 films) carries far more signal than "entertaining" (present in 6,500 films).

2. **UMAP Dimensionality Reduction** — Compress the 283-feature space (248 genome + ~23 genre + ~11 decade) down to 20 dimensions while preserving local neighborhood structure. This makes density-based clustering feasible.

3. **HDBSCAN Clustering** — Discover natural groupings in the 20D space without pre-specifying the number of clusters. Points in low-density regions are labeled as outliers rather than forced into bad fits.

4. **XGBoost Validation** — Train an independent supervised model to predict cluster labels. High cross-validated accuracy (>90%) confirms the clusters are real and learnable, not statistical noise.

5. **SHAP Interpretation** — For each cluster, SHAP values reveal exactly which features (and feature *interactions*) XGBoost uses to identify it. This tells us *what makes each rail distinctive*.

6. **Outlier Recovery** — XGBoost predicts cluster membership for the ~7% of films HDBSCAN couldn't place, with confidence scores. Films above 50% confidence become curator review candidates.

7. **LLM Naming** — A local LLM (Ollama) generates human-readable rail names using the SHAP feature profiles, producing names like "Nostalgic Coming-of-Age Dramas" instead of "Cluster 7".

---

### How to Read This Notebook

Every code cell has a **header block** at the top showing its inputs, what it does, and its outputs:

```python
# ╔═══════════════════════════════════════════╗
# ║  CELL PURPOSE                              ║
# ║  Inputs:  what this cell reads             ║
# ║  Outputs: what this cell produces          ║
# ╚═══════════════════════════════════════════╝
```

Every section has a **markdown explanation** above the code that teaches the *why* — not just what the code does, but why that technique was chosen and how it fits into the broader pipeline.

---

### Prerequisites

**Ollama** (for LLM naming in Section 9):
```bash
ollama pull llama3.2:3b
ollama serve
```

**XGBoost + SHAP** (for cluster validation in Section 8.5):
```bash
pip install xgboost shap
```

## Pipeline Overview

<div style="display:flex; align-items:center; justify-content:center; flex-wrap:wrap; gap:6px; padding:20px; background:linear-gradient(135deg, #1a1a2e 0%, #16213e 100%); border-radius:12px; margin:10px 0;">
  <div style="background:#2196F3; color:white; padding:14px 16px; border-radius:8px; text-align:center; min-width:110px; box-shadow:0 2px 8px rgba(33,150,243,0.3);">
    <b style="font-size:13px;">Raw Data</b><br><small style="opacity:0.9;">8,000 movies<br>248 genome tags</small>
  </div>
  <div style="font-size:22px; color:#64b5f6;">→</div>
  <div style="background:#4CAF50; color:white; padding:14px 16px; border-radius:8px; text-align:center; min-width:110px; box-shadow:0 2px 8px rgba(76,175,80,0.3);">
    <b style="font-size:13px;">IDF Weighting</b><br><small style="opacity:0.9;">Upweight rare<br>features</small>
  </div>
  <div style="font-size:22px; color:#64b5f6;">→</div>
  <div style="background:#FF9800; color:white; padding:14px 16px; border-radius:8px; text-align:center; min-width:110px; box-shadow:0 2px 8px rgba(255,152,0,0.3);">
    <b style="font-size:13px;">Genre + Decade</b><br><small style="opacity:0.9;">~34 binary<br>nudge signals</small>
  </div>
  <div style="font-size:22px; color:#64b5f6;">→</div>
  <div style="background:#9C27B0; color:white; padding:14px 16px; border-radius:8px; text-align:center; min-width:110px; box-shadow:0 2px 8px rgba(156,39,176,0.3);">
    <b style="font-size:13px;">UMAP</b><br><small style="opacity:0.9;">282D → 20D<br>non-linear</small>
  </div>
  <div style="font-size:22px; color:#64b5f6;">→</div>
  <div style="background:#F44336; color:white; padding:14px 16px; border-radius:8px; text-align:center; min-width:110px; box-shadow:0 2px 8px rgba(244,67,54,0.3);">
    <b style="font-size:13px;">HDBSCAN</b><br><small style="opacity:0.9;">Density-based<br>clustering</small>
  </div>
  <div style="font-size:22px; color:#64b5f6;">→</div>
  <div style="background:#00BCD4; color:white; padding:14px 16px; border-radius:8px; text-align:center; min-width:110px; box-shadow:0 2px 8px rgba(0,188,212,0.3);">
    <b style="font-size:13px;">XGBoost</b><br><small style="opacity:0.9;">Validate +<br>SHAP interpret</small>
  </div>
  <div style="font-size:22px; color:#64b5f6;">→</div>
  <div style="background:#607D8B; color:white; padding:14px 16px; border-radius:8px; text-align:center; min-width:110px; box-shadow:0 2px 8px rgba(96,125,139,0.3);">
    <b style="font-size:13px;">LLM Naming</b><br><small style="opacity:0.9;">SHAP-guided<br>rail names</small>
  </div>
</div>

## 1 — Configuration

All tuneable hyperparameters and file paths for the full pipeline. Centralising these at the top means you can experiment with different settings without hunting through the code — just change a value here and re-run.

The configuration is split into several groups:

**Sweep toggles** control whether the expensive hyperparameter searches run. Set these to `True` when you want to explore the parameter space; leave them `False` for a normal pipeline run with the already-tuned values.

**Data options** toggle whether genre and decade features are included. These act as gentle "nudge signals" alongside the genome backbone — you can disable them to see how much they contribute.

**UMAP settings** were tuned from a 216-combination sweep followed by a 25-combination targeted refinement. The comments in the code explain what each parameter does and why its value was chosen.

**HDBSCAN settings** were tuned from a 350-combination sweep. The key parameter `min_cluster_size=48` sits at a sweet spot in the DBCV landscape — go lower and cluster quality drops sharply.

**Outputs:** Global constants used by every subsequent cell: sweep toggles, data options, UMAP/HDBSCAN hyperparameters, LLM model settings, and file paths.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  CONFIGURATION — all tuneable parameters for the full pipeline             ║
# ║  Inputs:  None (constants only)                                            ║
# ║  Outputs: Global constants used by every subsequent cell                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Sweep Toggles ────────────────────────────────────────────────────────────
RUN_UMAP_SWEEP    = False   # Sweep UMAP n_components, n_neighbors, metric, min_dist
RUN_HDBSCAN_SWEEP = False   # Sweep HDBSCAN min_cluster_size, min_samples, epsilon, selection_method

# ── Data Options ─────────────────────────────────────────────────────────────
# INCLUDE_GENRE_FEATURES: Add one-hot genre columns (~23 binary nudge signals).
INCLUDE_GENRE_FEATURES  = True

# INCLUDE_DECADE_FEATURE: Add one-hot decade columns (~10 binary nudge signals).
INCLUDE_DECADE_FEATURE  = True

# ── UMAP Settings ────────────────────────────────────────────────────────────
# Tuned from 216-combo sweep + 25-combo targeted refinement.
# Key findings:
#   - n_components=20: trust saturated across 17-25; 20D is clean round number
#   - n_neighbors=60: refinement showed 60 > 100 for nn_overlap (0.569 vs 0.564 avg)
#     Trust identical (0.9866 vs 0.9866). Lower nn = faster fit + better local structure.
#   - min_dist=0.1: best nn_overlap; 0.0 marginal trust gain but worse overlap
#   - correlation: lowest reconstruction error; trust identical across metrics
UMAP_N_COMPONENTS       = 20
UMAP_N_NEIGHBORS        = 60
UMAP_MIN_DIST           = 0.1
UMAP_METRIC             = 'correlation'

# Visualisation UMAP (separate — always 3D for scatter plots)
UMAP_VIS_N_COMPONENTS   = 3
UMAP_VIS_N_NEIGHBORS    = 60
UMAP_VIS_MIN_DIST       = 0.1

# ── HDBSCAN Settings ────────────────────────────────────────────────────────
# Tuned from 350-combo sweep + targeted mcs refinement.
# Key findings:
#   - mcs=48 sweet spot: 41 clusters, DBCV=0.511 (best!), sil=0.598, prob=0.908
#   - Cliff at mcs=45: DBCV drops to 0.399, persistence crashes to 0.738
#   - ms=10 is the quality inflection point (DBCV +0.09 vs ms=3)
#   - EOM >> Leaf across all metrics
HDBSCAN_MIN_CLUSTER_SIZE = 48
HDBSCAN_MIN_SAMPLES      = 10
HDBSCAN_EPSILON          = 0.2
HDBSCAN_SELECTION_METHOD = 'eom'

# ── LLM Naming (Ollama) ─────────────────────────────────────────────────────
OLLAMA_MODEL         = 'llama3.2:3b'
OLLAMA_BASE_URL      = 'http://localhost:11434'

# ── Paths ────────────────────────────────────────────────────────────────────
GENOME_SCORES_PATH = 'feature_data_longform.csv'
GENOME_TAGS_PATH   = 'feature_taxonomy.csv'
OMDB_DATA_DIR      = 'extra_data'
RESULTS_DIR        = 'Dashboard/results'

print('Configuration loaded.')
print(f'  Preprocessing: IDF (sklearn TfidfTransformer) on genome features only')
print(f'  UMAP:          {UMAP_N_COMPONENTS}D, {UMAP_N_NEIGHBORS} neighbors, min_dist={UMAP_MIN_DIST}, {UMAP_METRIC}')
print(f'  HDBSCAN:       min_size={HDBSCAN_MIN_CLUSTER_SIZE}, min_samples={HDBSCAN_MIN_SAMPLES}, eps={HDBSCAN_EPSILON}, {HDBSCAN_SELECTION_METHOD}')
print(f'  Ollama:        {OLLAMA_MODEL} @ {OLLAMA_BASE_URL}')
print(f'  Sweeps:        UMAP={RUN_UMAP_SWEEP}, HDBSCAN={RUN_HDBSCAN_SWEEP}')

## 2 — Imports

Loads all core libraries used throughout the notebook. Here's what each one does:

- **numpy** — Fast array operations and linear algebra. Everything in the pipeline ultimately runs on numpy arrays.
- **pandas** — DataFrame operations for loading CSVs, pivoting tables, and filtering data.
- **umap-learn** — UMAP (Uniform Manifold Approximation and Projection) for non-linear dimensionality reduction. Compresses hundreds of features down to a clusterable low-dimensional space while preserving local neighborhood structure.
- **hdbscan** — Hierarchical Density-Based Spatial Clustering. Discovers clusters from the density landscape of the data without requiring a pre-set number of clusters.
- **sklearn.impute.SimpleImputer** — Safety net for filling any missing values (shouldn't be needed with our data, but prevents silent NaN propagation).

Later cells import additional libraries as needed (XGBoost, SHAP, matplotlib, plotly, etc.) — they're imported locally to make dependencies clear.

**Inputs:** None (standard library + pip-installed packages).

**Outputs:** `numpy`, `pandas`, `umap`, `hdbscan`, `Path`, `SimpleImputer` available in the global namespace.

In [35]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  IMPORTS — core libraries for the entire pipeline                          ║
# ║  Inputs:  None (standard + pip-installed packages)                         ║
# ║  Outputs: numpy, pandas, umap, hdbscan, Path, SimpleImputer in namespace  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import numpy as np                    # Array operations and linear algebra
import pandas as pd                   # DataFrames for tabular data manipulation
import warnings
warnings.filterwarnings('ignore')     # Suppress convergence/deprecation warnings

from pathlib import Path              # OS-agnostic file path handling
from sklearn.impute import SimpleImputer  # Fill missing values (safety net)
import umap                           # UMAP dimensionality reduction
import hdbscan                        # Density-based clustering

print('Imports ready')

Imports ready


## 3 — Load Genome Data (The "Spine")

The genome tags are the **backbone of the entire model**. Each of the 8,000 movies has been scored on **248 taxonomy tags** — things like *visually appealing*, *revenge*, *romantic*, *claymation*, *mind-bending* — on a continuous **0 → 3 relevance scale**.

These aren't binary yes/no labels. A score of 2.8 on "revenge" means the film is deeply driven by revenge; a score of 0.4 means there's a faint trace. This granularity is what gives the model its resolution — two "action" films can be completely different because one scores 2.9 on *stylized violence* and 0.1 on *thought-provoking*, while the other scores 0.3 and 2.7. Standard genre labels can't make that distinction.

The genome data arrives in **long form** (`movie × tag × relevance`) and gets pivoted into a wide **feature matrix** (`movies × 248 columns`). Duplicate tag names (like "The Hero's Journey" which appears twice with different `feature_id` values) are disambiguated by appending the ID.

**Inputs:** `GENOME_SCORES_PATH` → `feature_data_longform.csv`, `GENOME_TAGS_PATH` → `feature_taxonomy.csv`.

**Process:** Read CSVs, rename columns to standard names, merge taxonomy onto scores, disambiguate duplicate tag names, pivot from long-form to wide matrix.

**Outputs:** `feature_matrix` — a DataFrame of shape `(~8,000 movies × 248 genome features)`, indexed by `imdb_id`.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  LOAD GENOME DATA — pivot long-form scores into wide feature matrix        ║
# ║  Inputs:  GENOME_SCORES_PATH (CSV), GENOME_TAGS_PATH (CSV)                ║
# ║  Outputs: feature_matrix (DataFrame, ~8000 × 248)                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Read the long-form genome scores: one row per (movie, tag, relevance) triple
features_long = pd.read_csv(GENOME_SCORES_PATH)
# Read the tag taxonomy: maps feature_id → human-readable tag name
taxonomy = pd.read_csv(GENOME_TAGS_PATH)

# ── Standardise column names (handle both old and new CSV schemas) ────────────
if 'tagId' in features_long.columns:
    features_long = features_long.rename(columns={
        'tagId': 'feature_id', 'movieId': 'imdb_id', 'relevance': 'trigger'
    })
if 'tagId' in taxonomy.columns:
    taxonomy = taxonomy.rename(columns={'tagId': 'feature_id', 'tag': 'feature'})

# Merge taxonomy names onto the score table so each row has its tag name
features_long = features_long.merge(taxonomy, on='feature_id', how='left')

# ── Disambiguate duplicate tag names so every feature_id gets its own column ─
# "The Hero's Journey" appears twice (IDs 177 and 400) with different semantics
_name_counts = taxonomy['feature'].value_counts()
_dup_names = set(_name_counts[_name_counts > 1].index)

def _make_unique_name(row):
    """Append feature_id to disambiguate duplicate tag names."""
    if row['feature'] in _dup_names:
        return f"{row['feature']} ({row['feature_id']})"
    return row['feature']

features_long['feature_unique'] = features_long.apply(_make_unique_name, axis=1)

# ── Pivot: long-form → wide matrix (movies × features) ───────────────────────
# Each cell is the relevance score (0-3) for that movie × tag combination
feature_matrix = features_long.pivot_table(
    index='imdb_id',           # Rows = movies (IMDb IDs)
    columns='feature_unique',  # Columns = 248 unique tag names
    values='trigger',          # Cell values = relevance scores (0-3)
    aggfunc='first'            # Take first value (no duplicates expected)
).fillna(0)                    # Fill any missing movie×tag pairs with 0

print(f'Genome matrix: {feature_matrix.shape[0]:,} movies x {feature_matrix.shape[1]} features')

### 3.1 — Feature Audit: 248 Taxonomy Entries → 248 Genome Columns

**Why audit?** Data pipelines often silently drop or merge features during pivoting — especially when there are duplicate names. If a column gets lost, the model might still "work" but it'll be quietly using fewer dimensions than we think. This audit catches that.

The `feature_taxonomy.csv` file contains **248 rows** (features). The tag name **"The Hero's Journey"** appears twice with two different `feature_id` values:
- `feature_id 177`: The Hero's Journey (mean relevance ~0.32, present in ~1,958 movies)
- `feature_id 400`: The Hero's Journey (mean relevance ~0.57, present in ~3,638 movies)

These are genuinely different annotations — they only correlate at r ≈ 0.54 and have identical values for only ~65% of movies. This makes sense: one might capture the *structural arc* (hero leaves home → faces trials → returns transformed), while the other captures the *thematic feel* of a hero's journey story.

**Decision:** We keep both as separate columns by appending the `feature_id` in parentheses when names collide (e.g., `"The Hero's Journey (177)"` and `"The Hero's Journey (400)"`). All other 246 tags have unique names and are used as-is.

**Inputs:** `feature_matrix` (from the previous cell), `GENOME_TAGS_PATH` (re-read for verification).

**Process:** Count taxonomy rows, count unique tag names, identify duplicates, verify the pivoted matrix has exactly 248 columns.

**Outputs:** An assertion that `feature_matrix.shape[1] == 248`. If this fails, the pipeline halts with a clear error message — better to fail loudly here than produce silently wrong clusters downstream.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  FEATURE AUDIT — verify all 248 taxonomy entries survived the pivot        ║
# ║  Inputs:  feature_matrix (from Cell 7), GENOME_TAGS_PATH (re-read)        ║
# ║  Outputs: Assertion (halts pipeline if feature count doesn't match)        ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Feature Audit: verify all 248 taxonomy entries are preserved ─────────────
_tax_raw = pd.read_csv(GENOME_TAGS_PATH)
_tag_col = 'feature' if 'feature' in _tax_raw.columns else 'tag'
_id_col  = 'feature_id' if 'feature_id' in _tax_raw.columns else 'tagId'

print(f'Taxonomy rows:       {len(_tax_raw)}')
print(f'Unique tag names:    {_tax_raw[_tag_col].nunique()}')
print(f'Pivoted columns:     {feature_matrix.shape[1]}')

_dupes = _tax_raw[_tax_raw.duplicated(subset=[_tag_col], keep=False)]
if len(_dupes) > 0:
    print(f'\nDuplicate tag names (kept as separate columns):')
    for _, row in _dupes.iterrows():
        _fid = row[_id_col]
        _tag = row[_tag_col]
        _col_name = f'{_tag} ({_fid})'
        _mask = features_long['feature_id'] == _fid
        _vals = features_long.loc[_mask, 'trigger']
        _in_matrix = _col_name in feature_matrix.columns
        print(f'  ID {_fid:>5}: "{_col_name}" — mean={_vals.mean():.3f}, '
              f'nonzero={int((_vals > 0).sum()):,}, in_matrix={_in_matrix}')
    print(f'\n→ Both kept as separate columns via feature_id disambiguation.')

assert feature_matrix.shape[1] == len(_tax_raw), (
    f'Expected {len(_tax_raw)} columns but got {feature_matrix.shape[1]}'
)
print(f'\n✓ All {len(_tax_raw)} taxonomy features preserved as separate columns.')

## 3b — Load OMDB Data + Extract Metadata

The genome tags tell us what a movie *feels like*, but they don't tell us its title, year, rating, or language. This cell loads **OMDB (Open Movie Database)** data to fill those gaps. The metadata serves four distinct purposes downstream:

1. **Genre features (Section 4)** — OMDB's genre field ("Action, Comedy, Sci-Fi") gets one-hot encoded into ~23 binary columns that act as nudge signals in the feature matrix.
2. **Decade features (Section 4)** — The release year gets bucketed into decades (1980s, 1990s, ...) and one-hot encoded into ~11 binary columns.
3. **LLM naming prompts (Section 9)** — Movie titles are included in the Ollama prompt so the LLM can see representative films when naming each cluster.
4. **Dashboard display** — The Streamlit dashboard shows movie titles, posters, ratings, plots, etc.

The cell also builds `jordan_df` — a metadata table with **content categories** (Family/Teen/Mature/Unknown) derived from MPAA ratings. These categories don't feed into the clustering model, but they're useful for analyzing cluster demographics in the dashboard.

**Inputs:** `OMDB_DATA_DIR` path → looks for `omdb_movies.csv` or `omdb_movies.json`; also checks for pre-built `jordan_metadata.csv`.

**Process:**
1. Load OMDB data (CSV preferred, JSON fallback) into a `movies_data` dict keyed by IMDb ID.
2. Build `dataset_imdb_ids` / `tt_codes` aligned with `feature_matrix` row order.
3. Build `jordan_df`: per-movie metadata with year, rating, language, content category, decade, and composite labels.
4. Build `title_lookup`: simple `{imdb_id: title}` dict for LLM naming prompts.

**Outputs:** `movies_data` (dict), `tt_codes` (list), `jordan_df` (DataFrame), `title_lookup` (dict).

In [37]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  LOAD OMDB DATA — movie metadata for genre, decade, naming, dashboard     ║
# ║  Inputs:  OMDB_DATA_DIR (CSV or JSON), feature_matrix.index for alignment ║
# ║  Outputs: movies_data (dict), tt_codes (list), jordan_df, title_lookup    ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

from pathlib import Path

# ── Load OMDB movie data ────────────────────────────────────────────────────
omdb_dir = Path(OMDB_DATA_DIR)

# Support both CSV and JSON formats
movies_csv = omdb_dir / 'omdb_movies.csv'
movies_json = omdb_dir / 'omdb_movies.json'

if movies_csv.exists():
    _omdb_df = pd.read_csv(movies_csv, dtype={'imdb_id': str})
    # Convert CSV rows to dict keyed by imdb_id (same structure as JSON)
    movies_data = {}
    for _, row in _omdb_df.iterrows():
        mid = str(row['imdb_id'])
        movies_data[mid] = row.to_dict()
    print(f'OMDB data loaded (CSV): {len(movies_data):,} movies')
elif movies_json.exists():
    import json as _json_loader
    with open(movies_json) as f:
        movies_data = _json_loader.load(f)
    print(f'OMDB data loaded (JSON): {len(movies_data):,} movies')
else:
    movies_data = {}
    print(f'⚠ OMDB data not found in {omdb_dir} — genre/decade/naming will be limited')

# ── Build dataset IMDb ID list (aligned with feature matrix) ─────────────────
dataset_imdb_ids = [str(mid) for mid in feature_matrix.index.tolist()]
tt_codes = dataset_imdb_ids

# ── Extract Jordan's metadata ───────────────────────────────────────────────
# Prefer pre-built jordan_metadata.csv if available
jordan_csv = omdb_dir / 'jordan_metadata.csv'
if jordan_csv.exists():
    jordan_df = pd.read_csv(jordan_csv, dtype={'imdb_id': str})
    print(f'Jordan metadata loaded from CSV: {jordan_df.shape[0]:,} rows')
else:
    # Build from OMDB responses
    jordan_rows = []
    for imdb_id in dataset_imdb_ids:
        movie = movies_data.get(imdb_id, {})
        if not movie:
            jordan_rows.append({
                'imdb_id': imdb_id,
                'year': 'Unknown', 'rating': 'Unknown', 'language': 'Unknown',
                'content_category': 'Unknown', 'decade': 0,
                'label_lang_rating': 'Unknown - Unknown',
                'label_lang_rating_decade': 'Unknown - Unknown - Unknown Decade'
            })
            continue

        year_raw = movie.get('Year', 'Unknown')
        rating   = movie.get('Rated', 'Unknown')
        if rating == 'N/A' or (isinstance(rating, float) and pd.isna(rating)):
            rating = 'Unknown'
        lang_raw = movie.get('Language', 'Unknown')
        lang = str(lang_raw).split(',')[0].strip() if isinstance(lang_raw, str) else 'Unknown'

        try:
            year_clean = int(str(year_raw)[:4])
        except (ValueError, TypeError):
            year_clean = 0

        rating_map = {
            'G': 'Family', 'PG': 'Family', 'TV-G': 'Family', 'TV-Y': 'Family', 'TV-Y7': 'Family',
            'PG-13': 'Teen', '12A': 'Teen', 'TV-14': 'Teen', 'TV-PG': 'Teen', 'TV-Y7-FV': 'Teen',
            'R': 'Mature', 'NC-17': 'Mature', 'TV-MA': 'Mature', 'X': 'Mature',
            'Not Rated': 'Unknown', 'Unrated': 'Unknown', 'N/A': 'Unknown', 'Unknown': 'Unknown',
            'Approved': 'Unknown', 'Passed': 'Unknown', 'GP': 'Unknown', 'M': 'Unknown', 'M/PG': 'Unknown',
        }
        content_category = rating_map.get(rating, 'Unknown')
        decade = (year_clean // 10) * 10

        label_lr = f'{lang} - {content_category}'
        label_lrd = f'{lang} - {content_category} - {decade}s' if decade > 0 else f'{lang} - {content_category} - Unknown Decade'

        jordan_rows.append({
            'imdb_id': imdb_id,
            'year': year_raw, 'rating': rating, 'language': lang,
            'content_category': content_category, 'decade': decade,
            'label_lang_rating': label_lr,
            'label_lang_rating_decade': label_lrd
        })

    jordan_df = pd.DataFrame(jordan_rows)
    # Save for reuse
    if omdb_dir.exists():
        jordan_df.to_csv(jordan_csv, index=False)
    print(f'Jordan metadata built from OMDB: {jordan_df.shape[0]:,} rows')

# ── Build title lookup for LLM naming prompts ───────────────────────────────
title_lookup = {}
for imdb_id, movie in movies_data.items():
    title = movie.get('Title', '')
    if isinstance(title, str) and title:
        title_lookup[imdb_id] = title

print(f'  Rating categories: {jordan_df["content_category"].value_counts().to_dict()}')
print(f'  Decade range: {sorted(int(d) for d in jordan_df[jordan_df["decade"] > 0]["decade"].unique())}')
print(f'  Title lookup: {len(title_lookup):,} movies with titles')

OMDB data loaded (CSV): 7,992 movies
Jordan metadata loaded from CSV: 8,000 rows
  Rating categories: {'Unknown': 3049, 'Mature': 2495, 'Teen': 1478, 'Family': 978}
  Decade range: [1900, 1910, 1920, 1930, 1940, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020]
  Title lookup: 7,992 movies with titles


## 4 — Optional Features: Genre + Decade (The "Bumps")

The 248 genome tags capture *what a movie feels like* — but they don't directly encode **genre** or **era**. Two films might have identical genome profiles but one is a 1970s Western and the other is a 2020s sci-fi. The genome alone can't see that difference.

So we add **~23 genre columns** and **~11 decade columns** as binary (0/1) **nudge signals** — small "bumps" that gently push same-genre or same-era films closer together in the feature space without overpowering the genome spine.

<div style="display:flex; align-items:center; justify-content:center; gap:10px; padding:18px; background:linear-gradient(135deg, #fff3e0 0%, #ffe0b2 100%); border-radius:12px; margin:10px 0;">
  <div style="text-align:center; padding:14px 18px; background:white; border-radius:8px; border:2px solid #FF9800; min-width:140px;">
    <b style="font-size:15px; color:#E65100;">248 Genome Tags</b><br>
    <small style="color:#666;">Continuous: 0 → 3</small><br>
    <span style="font-size:20px;">🦴</span><br>
    <small style="color:#888;"><b>THE SPINE</b><br>Carries ~95% of signal</small>
  </div>
  <div style="font-size:22px; color:#FF9800;">+</div>
  <div style="text-align:center; padding:14px 18px; background:white; border-radius:8px; border:2px solid #FF9800; min-width:120px;">
    <b style="font-size:15px; color:#E65100;">~23 Genres</b><br>
    <small style="color:#666;">Binary: 0 or 1</small><br>
    <span style="font-size:20px;">🏷️</span><br>
    <small style="color:#888;"><b>BUMPS</b><br>Action, Comedy,<br>Horror, etc.</small>
  </div>
  <div style="font-size:22px; color:#FF9800;">+</div>
  <div style="text-align:center; padding:14px 18px; background:white; border-radius:8px; border:2px solid #FF9800; min-width:120px;">
    <b style="font-size:15px; color:#E65100;">~11 Decades</b><br>
    <small style="color:#666;">Binary: 0 or 1</small><br>
    <span style="font-size:20px;">📅</span><br>
    <small style="color:#888;"><b>BUMPS</b><br>1950s, 1960s,<br>..., 2020s</small>
  </div>
  <div style="font-size:22px; color:#FF9800;">=</div>
  <div style="text-align:center; padding:14px 18px; background:white; border-radius:8px; border:2px solid #FF9800; min-width:140px;">
    <b style="font-size:15px; color:#E65100;">~282 Features</b><br>
    <small style="color:#666;">Mixed scales</small><br>
    <span style="font-size:20px;">📊</span><br>
    <small style="color:#888;"><b>COMBINED</b><br>Ready for IDF<br>weighting</small>
  </div>
</div>

> **Why binary?** Genre and decade are categorical, not continuous — a movie either *is* from the 1990s or it *isn't*. Keeping them as 0/1 signals means they can only nudge the distance calculation, never dominate it. The genome spine still drives the clustering.

In [38]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  GENRE + DECADE FEATURES — binary "bump" signals appended to genome       ║
# ║  Inputs:  feature_matrix, movies_data, jordan_df, config toggles          ║
# ║  Outputs: feature_matrix (now ~283 cols), X (numpy array), feature_names, ║
# ║           genome_cols, genre_cols, decade_cols                             ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Genre one-hot ────────────────────────────────────────────────────────────
if INCLUDE_GENRE_FEATURES:
    # Build genre one-hot from movies_data (loaded in previous cell — works for both CSV and JSON)
    genre_rows = []
    for imdb_id, entry in movies_data.items():
        genres = entry.get('Genre', '')
        if isinstance(genres, str) and genres and genres != 'N/A':
            for g in genres.split(','):
                genre_rows.append({'imdb_id': imdb_id, 'genre': g.strip()})

    if genre_rows:
        genre_df = pd.DataFrame(genre_rows)
        genre_df['_val'] = 1
        genre_onehot = genre_df.pivot_table(
            index='imdb_id', columns='genre', values='_val', aggfunc='max'
        ).fillna(0)
        genre_onehot.columns = [f'genre_{c}' for c in genre_onehot.columns]

        existing = [c for c in feature_matrix.columns if c.startswith('genre_')]
        if existing:
            feature_matrix = feature_matrix.drop(columns=existing)
        feature_matrix = feature_matrix.join(genre_onehot, how='left').fillna(0)
        print(f'Genre features added: {len(genre_onehot.columns)} columns')
    else:
        print('No genre data found in OMDB — skipping')
else:
    existing = [c for c in feature_matrix.columns if c.startswith('genre_')]
    if existing:
        feature_matrix = feature_matrix.drop(columns=existing)
    print('Genre features: SKIPPED')

# ── Decade one-hot ───────────────────────────────────────────────────────────
# Each decade gets its own binary column (decade_1920s, decade_1930s, ...).
# No ordinal assumption — 1980s is not "closer to" 1990s than 1960s.
existing_decade = [c for c in feature_matrix.columns
                   if c.startswith('decade_') or c == 'meta_decade']
if existing_decade:
    feature_matrix = feature_matrix.drop(columns=existing_decade)

if INCLUDE_DECADE_FEATURE:
    jf = jordan_df.copy()
    jf['imdb_id'] = jf['imdb_id'].astype(str)
    jf = jf.set_index('imdb_id')[['decade']]
    jf = jf[jf['decade'] > 0]  # drop rows with no decade info

    # Create one-hot columns
    decade_onehot = pd.get_dummies(jf['decade'].astype(int), prefix='decade')
    # Rename columns to be human-readable: decade_1980 → decade_1980s
    decade_onehot.columns = [f'{c}s' for c in decade_onehot.columns]
    decade_onehot.index.name = 'imdb_id'

    feature_matrix = feature_matrix.join(decade_onehot, how='left').fillna(0)
    decade_col_names = sorted(decade_onehot.columns.tolist())
    print(f'Decade features added: {len(decade_col_names)} one-hot columns')
    print(f'  Decades: {", ".join(decade_col_names)}')
else:
    print('Decade feature: SKIPPED')

# ── Identify feature types and build matrix ──────────────────────────────────
genome_cols  = [c for c in feature_matrix.columns
                if not c.startswith('genre_') and not c.startswith('decade_')]
genre_cols   = [c for c in feature_matrix.columns if c.startswith('genre_')]
decade_cols  = [c for c in feature_matrix.columns if c.startswith('decade_')]

# Drop constant features
constant_cols = [c for c in feature_matrix.columns if feature_matrix[c].nunique() <= 1]
if constant_cols:
    print(f'Dropping {len(constant_cols)} constant features: {constant_cols}')
    feature_matrix = feature_matrix.drop(columns=constant_cols)
    genome_cols = [c for c in genome_cols if c not in constant_cols]
    genre_cols  = [c for c in genre_cols if c not in constant_cols]
    decade_cols = [c for c in decade_cols if c not in constant_cols]

X = feature_matrix.values.astype(np.float32)
feature_names = feature_matrix.columns.tolist()

print(f'\nFeature matrix: {feature_matrix.shape[0]:,} x {feature_matrix.shape[1]}')
print(f'  Genome: {len(genome_cols)}, Genre: {len(genre_cols)}, Decade: {len(decade_cols)}')

Genre features added: 23 columns
Decade features added: 13 one-hot columns
  Decades: decade_1900s, decade_1910s, decade_1920s, decade_1930s, decade_1940s, decade_1950s, decade_1960s, decade_1970s, decade_1980s, decade_1990s, decade_2000s, decade_2010s, decade_2020s

Feature matrix: 8,000 x 283
  Genome: 247, Genre: 23, Decade: 13


## 5 — IDF Weighting

### What is IDF?

IDF stands for **Inverse Document Frequency** — a technique borrowed from information retrieval (search engines). In text search, words like "the" and "is" appear in every document and carry no search value, while rare words like "cryptocurrency" or "paleontology" are highly discriminative. IDF downweights the common words and upweights the rare ones.

We apply the same logic to genome tags. A tag like **"entertaining"** scores > 0 in nearly every movie — it doesn't help distinguish one film from another, so it gets **downweighted**. A tag like **"claymation"** only scores highly in ~45 films — when a movie has it, that's a powerful distinguishing signal, so it gets **upweighted**.

### The Formula

For each genome feature *t*, sklearn's `TfidfTransformer` computes:

> **idf(t) = log((1 + n) / (1 + df(t))) + 1**

Where:
- **n** = total number of movies (~8,000)
- **df(t)** = number of movies where tag *t* has a nonzero score

The "+1" smoothing prevents division by zero and ensures even the rarest features don't get infinitely large weights.

### Why Only Genome Features?

We only IDF-weight the **248 genome columns** (continuous 0–3 scores). The ~34 genre and decade columns are **binary** (0 or 1), so applying IDF would just multiply each column by a single constant — it wouldn't change the relative distances between movies. Those binary "bumps" pass through unchanged and get concatenated back after IDF.

### Why Not StandardScaler + TruncatedSVD?

We tested SVD (150 components) as an alternative to IDF. The UMAP parameter sweep showed a significant quality drop: **trustworthiness fell from 0.987 → 0.918** and **neighborhood overlap from 0.569 → 0.387**. SVD optimizes for *global variance* (keeping the dimensions that explain the most spread), but UMAP needs *local neighborhood signals* (which nearby points are similar). IDF preserves the full 283-dimensional feature space and simply reweights by informativeness — giving UMAP the richest possible input.

<div style="padding:18px; background:linear-gradient(135deg, #e8f5e9 0%, #c8e6c9 100%); border-radius:12px; margin:10px 0;">

<div style="display:flex; align-items:stretch; justify-content:center; gap:16px; flex-wrap:wrap;">
  <div style="flex:1; min-width:250px; padding:14px 18px; background:white; border-radius:8px; border:2px solid #4CAF50;">
    <b style="font-size:14px; color:#2E7D32;">🦴 Genome Features (248 cols) — IDF APPLIED</b><br><br>
    <small style="color:#666;">
      The spine gets <b>reweighted by IDF</b>. Tags that appear across nearly every film (like "entertaining") get <b>downweighted</b> — they don't help distinguish one movie from another.
      Tags that appear in only a handful of films (like "claymation" or "mumblecore") get <b>upweighted</b> — they're the rare signals that make clusters distinctive.
    </small>
    <div style="display:flex; gap:10px; margin-top:12px;">
      <div style="text-align:center; padding:8px; background:#FFEBEE; border-radius:6px; flex:1;">
        <b style="color:#F44336;">Low IDF</b><br>
        <small>"romance"<br>6,500 / 8,000 films<br>→ downweighted</small>
      </div>
      <div style="text-align:center; padding:8px; background:#E3F2FD; border-radius:6px; flex:1;">
        <b style="color:#2196F3;">High IDF</b><br>
        <small>"claymation"<br>45 / 8,000 films<br>→ upweighted</small>
      </div>
    </div>
  </div>

  <div style="flex:1; min-width:250px; padding:14px 18px; background:white; border-radius:8px; border:2px solid #FF9800;">
    <b style="font-size:14px; color:#E65100;">🏷️📅 Genre + Decade (~34 cols) — LEFT RAW</b><br><br>
    <small style="color:#666;">
      The bumps are <b>not</b> IDF-weighted. They're already binary (0 or 1), so IDF would either leave them unchanged or distort them. They stay as raw nudge signals — small enough to influence clustering without overpowering the genome, but present enough to catch genre/era patterns the genome misses.
    </small>
    <div style="margin-top:12px; text-align:center; padding:8px; background:#FFF3E0; border-radius:6px;">
      <small>Binary features pass through unchanged:<br><b>0</b> (not this genre/decade) or <b>1</b> (yes, this genre/decade)</small>
    </div>
  </div>
</div>

<div style="margin-top:14px; padding:10px 14px; background:white; border-radius:8px; text-align:center; border:1px solid #A5D6A7;">
  <small style="color:#333;">
    <b>After IDF:</b>&nbsp;&nbsp;
    <span style="color:#2E7D32;">248 IDF-weighted genome features</span> &nbsp;+&nbsp;
    <span style="color:#E65100;">~23 raw genre</span> &nbsp;+&nbsp;
    <span style="color:#E65100;">~11 raw decade</span> &nbsp;=&nbsp;
    <b>~282 features → UMAP</b>
  </small>
</div>

</div>

> **The key design choice:** IDF only touches the genome spine because those are the continuous relevance scores where frequency-based reweighting makes sense. The binary bumps are already on a 0/1 scale — applying IDF to them would just multiply each column by a single constant, which doesn't change the relative distances between movies. So they pass through untouched.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  IDF WEIGHTING — reweight genome features by rarity, leave binary as-is   ║
# ║  Inputs:  X (raw feature matrix), genome_cols, genre_cols, decade_cols,   ║
# ║           feature_names                                                    ║
# ║  Outputs: X_weighted (IDF-reweighted matrix), idf_weights (248 weights)   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

from sklearn.feature_extraction.text import TfidfTransformer

# ── Split matrix by feature type ────────────────────────────────────────────
# We only apply IDF to genome columns — binary genre/decade columns would just
# get multiplied by a constant, which doesn't change relative distances
genome_idx = [feature_names.index(c) for c in genome_cols]   # Indices of 248 genome cols
genre_idx  = [feature_names.index(c) for c in genre_cols]    # Indices of ~23 genre cols
decade_idx = [feature_names.index(c) for c in decade_cols]   # Indices of ~11 decade cols

X_genome = X[:, genome_idx]   # Shape: (n_movies, 248) — continuous 0-3 relevance
X_genre  = X[:, genre_idx]    # Shape: (n_movies, ~23) — binary 0/1
X_decade = X[:, decade_idx]   # Shape: (n_movies, ~11) — binary 0/1

# ── Apply IDF to genome features only ───────────────────────────────────────
# TfidfTransformer with use_idf=True computes IDF weights from the data:
#   idf(t) = log((1 + n) / (1 + df(t))) + 1
# where df(t) = number of movies where tag t has a nonzero score.
# Rare tags (few movies) get HIGH weights; common tags get LOW weights.
tfidf = TfidfTransformer(use_idf=True, smooth_idf=True, sublinear_tf=False)
X_genome_weighted = tfidf.fit_transform(X_genome).toarray().astype(np.float32)
idf_weights = tfidf.idf_   # Array of 248 IDF weights (one per genome feature)

print(f'IDF weighting applied to {len(genome_cols)} genome features')
print(f'  IDF weight range: {idf_weights.min():.3f} – {idf_weights.max():.3f}')
print(f'  Genome weighted shape: {X_genome_weighted.shape}')

# ── Concatenate: weighted genomes + raw binary features ─────────────────────
# The final matrix has IDF-weighted genome columns followed by untouched binary columns
X_weighted = np.hstack([X_genome_weighted, X_genre, X_decade])

print(f'\nFinal weighted matrix: {X_weighted.shape}')
print(f'  Genome (IDF): {X_genome_weighted.shape[1]}, Genre (raw): {X_genre.shape[1]}, Decade (raw): {X_decade.shape[1]}')

### 5-viz — IDF Weight Distribution

This visualisation answers: **how much did IDF change the feature landscape?**

**Left panel (histogram):** Shows the distribution of all 248 IDF weights. If most weights cluster near the median, IDF is making small adjustments. If there's a long right tail, some features are being dramatically upweighted — those are the rare, distinctive tags that will drive cluster formation.

**Right panel (bar chart):** The 15 features with the **highest IDF** (red bars, rarest) and the 15 with the **lowest IDF** (blue bars, most common). This tells you which specific tags the pipeline considers most and least informative. You'd expect high-IDF features to be niche genres or production techniques (claymation, mumblecore, avant-garde), and low-IDF features to be broad qualities (entertaining, interesting, good acting).

**Inputs:** `idf_weights` (array of 248 floats), `genome_cols` (list of 248 feature names).

**Process:** Histogram of all weights + sorted horizontal bar chart of the 30 most extreme features.

**Outputs:** Matplotlib figure (displayed inline). No new variables created.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  IDF VISUALISATION — histogram + top/bottom features bar chart             ║
# ║  Inputs:  idf_weights (248 floats), genome_cols (248 names)               ║
# ║  Outputs: Matplotlib figure (inline display only)                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import matplotlib.pyplot as plt

# ── Visualisation: IDF Weight Distribution ───────────────────────────────────
_fig, _axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: histogram of all 248 IDF weights
_axes[0].hist(idf_weights, bins=40, color='#4CAF50', edgecolor='white', alpha=0.9)
_axes[0].axvline(np.median(idf_weights), color='#F44336', linestyle='--', linewidth=2,
                 label=f'Median: {np.median(idf_weights):.2f}')
_axes[0].set_xlabel('IDF Weight', fontsize=13)
_axes[0].set_ylabel('Number of Features', fontsize=13)
_axes[0].set_title('Distribution of IDF Weights Across 248 Genome Features',
                    fontsize=14, fontweight='bold', pad=12)
_axes[0].legend(fontsize=12)
_axes[0].spines[['top', 'right']].set_visible(False)

# Right: top 15 rarest + bottom 15 most common
_sorted_idx = np.argsort(idf_weights)
_top15 = _sorted_idx[-15:][::-1]
_bot15 = _sorted_idx[:15]
_combined = np.concatenate([_top15, _bot15])
_names = [genome_cols[i] for i in _combined]
_vals = idf_weights[_combined]
_colors = ['#F44336'] * 15 + ['#2196F3'] * 15

_bars = _axes[1].barh(range(30), _vals, color=_colors, edgecolor='white', linewidth=0.5)
_axes[1].set_yticks(range(30))
_axes[1].set_yticklabels(_names, fontsize=9)
_axes[1].set_xlabel('IDF Weight', fontsize=13)
_axes[1].set_title('Top 15 Rarest (red) vs 15 Most Common (blue)',
                    fontsize=14, fontweight='bold', pad=12)
_axes[1].invert_yaxis()
_axes[1].spines[['top', 'right']].set_visible(False)

# Add value labels
for _bar, _v in zip(_bars, _vals):
    _axes[1].text(_bar.get_width() + 0.01, _bar.get_y() + _bar.get_height()/2,
                  f'{_v:.2f}', va='center', fontsize=8, color='#333')

_fig.tight_layout()
plt.show()

## 7 — UMAP Dimensionality Reduction

### The Curse of Dimensionality

Our feature matrix has **283 dimensions** (248 genome + ~23 genre + ~11 decade). In high-dimensional spaces, an unintuitive thing happens: **all distances become similar**. The farthest point and the nearest point end up almost the same distance away. This means density-based clustering (which relies on distance differences) can't find meaningful structure.

UMAP solves this by projecting the data down to a lower-dimensional space where the neighborhood relationships are preserved.

### How UMAP Works (Intuition)

UMAP operates in two phases:

1. **Build a neighborhood graph** — For each movie, find its *k* nearest neighbors in the original 283D space (using correlation distance). This creates a fuzzy topological representation of the data's shape.

2. **Optimize a low-dimensional layout** — Place the movies in the target space (20D for clustering, 3D for visualisation) and iteratively adjust positions so that the neighborhood graph is preserved as well as possible. Movies that were close in 283D should be close in 20D; movies that were far apart should stay far apart.

### Key Parameters

- **n_components (20):** The output dimensionality. Higher values preserve more structure but make clustering harder. 20D was chosen because trustworthiness saturated across 17–25 components in our sweep.
- **n_neighbors (60):** How many neighbors define "local" structure. Low values (5–15) preserve fine-grained local clusters; high values (100+) preserve more global structure. 60 was a sweep-optimized balance.
- **min_dist (0.1):** How tightly points can pack together. 0.0 allows tight clumps (good for clustering); higher values spread points out (better for visualization). 0.1 gave the best neighborhood overlap in our sweep.
- **metric ('correlation'):** How distance is measured in the original space. Correlation distance captures the *shape* of a movie's tag profile rather than its magnitude — two movies with the same relative tag rankings are considered similar even if one has uniformly higher scores.

### Two Separate Embeddings

We fit UMAP twice: once at **20D** (for HDBSCAN clustering — higher dimensionality preserves more structure) and once at **3D** (for scatter plot visualisation in the notebook and dashboard). These are independent fits, not projections of each other.

**Inputs:** `X_weighted` (shape `~8,000 × 283`), UMAP hyperparameters from config.

**Process:** Fit two independent UMAP models. The 2D slice is the first two components of the 3D embedding.

**Outputs:** `embedding_cluster` (20D, for clustering), `embedding_3d` (3D, for visualisation), `embedding_2d` (2D slice).

<div style="display:flex; align-items:center; justify-content:center; gap:16px; padding:18px; background:linear-gradient(135deg, #f3e5f5 0%, #e1bee7 100%); border-radius:12px; margin:10px 0;">
  <div style="text-align:center; padding:14px 20px; background:white; border-radius:8px; border:2px solid #9C27B0;">
    <b style="font-size:15px;">282 Dimensions</b><br>
    <small style="color:#666;">Each movie is a point in<br>282-dimensional space<br>(genome + genre + decade)</small>
  </div>
  <div style="text-align:center;">
    <div style="font-size:28px; color:#9C27B0;">→ UMAP →</div>
    <small style="color:#7B1FA2;">Preserve local<br>neighborhoods</small>
  </div>
  <div style="text-align:center; padding:14px 20px; background:white; border-radius:8px; border:2px solid #9C27B0;">
    <b style="font-size:15px;">20 Dimensions</b><br>
    <small style="color:#666;">Similar movies stay close,<br>dissimilar movies stay far,<br>but now in a clusterable space</small>
  </div>
</div>

> **Why UMAP?** High-dimensional data is sparse — distances become meaningless ("curse of dimensionality"). UMAP learns a non-linear mapping that preserves the *neighborhood structure*: if two movies were similar in 282D, they'll be close in 20D. This makes density-based clustering (HDBSCAN) possible.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  UMAP DIMENSIONALITY REDUCTION — 283D → 20D (clustering) + 3D (viz)       ║
# ║  Inputs:  X_weighted (~8000 × 283), UMAP config params                    ║
# ║  Outputs: embedding_cluster (20D), embedding_3d (3D), embedding_2d (2D)   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print(f'UMAP ({UMAP_METRIC}) on IDF-weighted features ({X_weighted.shape[1]} dims)...')

# ── Clustering embedding (20D) ───────────────────────────────────────────────
# Higher dimensionality preserves more fine-grained structure for HDBSCAN.
# low_memory=False is faster for <10K samples.
reducer_cluster = umap.UMAP(
    n_components=UMAP_N_COMPONENTS,     # 20D output
    n_neighbors=UMAP_N_NEIGHBORS,       # 60 — local vs global balance
    min_dist=UMAP_MIN_DIST,             # 0.1 — how tightly points can pack
    metric=UMAP_METRIC,                 # 'correlation' — best for tag profiles
    random_state=42,                    # Reproducibility
    low_memory=False                    # Faster for small datasets
)

# ── Visualisation embedding (3D) ─────────────────────────────────────────────
# Separate UMAP fit specifically for 3D scatter plots in the notebook and dashboard.
# Uses same metric and neighbors but only 3 output dimensions.
reducer_3d = umap.UMAP(
    n_components=UMAP_VIS_N_COMPONENTS, # 3D output
    n_neighbors=UMAP_VIS_N_NEIGHBORS,   # 60
    min_dist=UMAP_VIS_MIN_DIST,         # 0.1
    metric=UMAP_METRIC,                 # 'correlation'
    random_state=42
)

print(f'  Fitting clustering embedding ({UMAP_N_COMPONENTS}D)...')
embedding_cluster = reducer_cluster.fit_transform(X_weighted)  # Shape: (n_movies, 20)

print(f'  Fitting visualisation embedding ({UMAP_VIS_N_COMPONENTS}D)...')
embedding_3d = reducer_3d.fit_transform(X_weighted)            # Shape: (n_movies, 3)
embedding_2d = embedding_3d[:, :2]                             # Shape: (n_movies, 2) — first 2 components

print(f'\n  Clustering: {embedding_cluster.shape}')
print(f'  Visualisation: {embedding_3d.shape}')

### 7-viz — UMAP 3D Embedding

**What to look for:** Natural clusters should appear as visible dense blobs separated by gaps or lower-density regions. If the 3D projection looks like a uniform cloud with no structure, UMAP may not be capturing meaningful patterns (or the data genuinely lacks clear groupings).

Keep in mind this is a **3D projection** of what HDBSCAN sees as a 20D space — the actual clustering embedding preserves much more structure than what's visible here. Think of this as a "shadows on the wall" view: real structure that's separated in 20D might overlap when squished to 3D.

**Inputs:** `embedding_3d` (shape `~8,000 × 3`).

**Process:** 3D scatter plot with matplotlib, coloured uniformly purple (no cluster labels yet — this is a pre-clustering sanity check).

**Outputs:** Matplotlib 3D figure (displayed inline). No new variables created.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  UMAP 3D VISUALISATION — pre-clustering structure check                    ║
# ║  Inputs:  embedding_3d (n_movies × 3)                                     ║
# ║  Outputs: Matplotlib 3D figure (inline display only)                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Visualisation: 3D UMAP Embedding ─────────────────────────────────────────
from mpl_toolkits.mplot3d import Axes3D

_fig = plt.figure(figsize=(14, 9))
_ax = _fig.add_subplot(111, projection='3d')

_ax.scatter(
    embedding_3d[:, 0], embedding_3d[:, 1], embedding_3d[:, 2],
    s=4, alpha=0.35, c='#9C27B0', edgecolors='none'
)

_ax.set_title('3D UMAP Embedding — Pre-Clustering Structure',
              fontsize=15, fontweight='bold', pad=20)
_ax.set_xlabel('UMAP-1', fontsize=11, labelpad=8)
_ax.set_ylabel('UMAP-2', fontsize=11, labelpad=8)
_ax.set_zlabel('UMAP-3', fontsize=11, labelpad=8)
_ax.tick_params(labelsize=8)
_ax.view_init(elev=25, azim=135)

_fig.tight_layout()
plt.show()
print(f'  {embedding_3d.shape[0]:,} movies projected to 3D for visualisation')
print(f'  Clustering uses separate {UMAP_N_COMPONENTS}D embedding')

### 7.1 — UMAP Parameter Sweep

UMAP has several hyperparameters that significantly affect the quality of the embedding. Rather than guessing, we run an **exhaustive grid search** over **216 combinations** and measure three complementary quality metrics. The sweep is controlled by `RUN_UMAP_SWEEP = True/False` in the config.

### The Three Metrics

**Trustworthiness (higher = better):** Measures whether points that are close together in the embedding were *actually* close in the original space. High trustworthiness (>0.95) means the embedding isn't creating false neighborhoods — when UMAP says two movies are similar, they really are. This is the most important metric because HDBSCAN clusters based on proximity.

**Reconstruction error (lower = better):** Measures overall distance distortion between the original and embedded spaces. Unlike trustworthiness (which only checks local neighborhoods), this captures global distance preservation. Computed on a subsample of pairwise distances to keep it tractable.

**Neighborhood overlap, k=15 (higher = better):** For each movie, what fraction of its 15 nearest neighbors in the original space are still among its 15 nearest neighbors in the embedding? This is the most direct measure of local structure preservation. An overlap of 0.5 means half of each movie's original neighbors are preserved — quite good for a compression from 283D to 20D.

### The Grid

- **n_components:** [10, 15, 20, 30, 50, 80] — How many output dimensions?
- **n_neighbors:** [15, 30, 50, 100] — How "local" is the neighborhood definition?
- **metric:** ['correlation', 'cosine', 'euclidean'] — How is distance measured?
- **min_dist:** [0.0, 0.05, 0.1] — How tightly can points cluster?

**Inputs:** `X_weighted` (full feature matrix), `RUN_UMAP_SWEEP` toggle.

**Process:** For each of 216 combinations, fit UMAP, compute all three metrics (subsampled for speed: 2,000 movies for trustworthiness/overlap, 500 for pairwise distances). Results printed as a live table with per-metric and per-parameter aggregates.

**Outputs:** `umap_sweep_results` (list of dicts), `umap_sweep_df` (DataFrame). Skipped if `RUN_UMAP_SWEEP = False`.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  UMAP PARAMETER SWEEP — 216-combo grid search for optimal UMAP config     ║
# ║  Inputs:  X_weighted, RUN_UMAP_SWEEP toggle                               ║
# ║  Outputs: umap_sweep_results (list), umap_sweep_df (DataFrame)            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

if RUN_UMAP_SWEEP:
    from itertools import product as _product
    from sklearn.manifold import trustworthiness as _tw
    from sklearn.neighbors import NearestNeighbors as _NN

    _nc_list = [10, 15, 20, 30, 50, 80]
    _nn_list = [15, 30, 50, 100]
    _um_list = ['correlation', 'cosine', 'euclidean']
    _md_list = [0.0, 0.05, 0.1]

    # Pre-compute original-space neighbours for overlap metric (subsample for speed)
    _sub_n = min(2000, X_weighted.shape[0])
    _sub_idx = np.random.RandomState(42).choice(X_weighted.shape[0], _sub_n, replace=False)
    _sub_X = X_weighted[_sub_idx]
    _k_overlap = 15
    _nn_orig = _NN(n_neighbors=_k_overlap + 1, metric='correlation').fit(_sub_X)
    _orig_neighbors = _nn_orig.kneighbors(_sub_X, return_distance=False)[:, 1:]  # exclude self

    umap_sweep_results = []
    _total = len(_nc_list) * len(_nn_list) * len(_um_list) * len(_md_list)
    print(f'UMAP sweep: {len(_nc_list)} n_comp x {len(_nn_list)} n_neigh x {len(_um_list)} metrics x {len(_md_list)} min_dist = {_total} combos')
    print(f'  Embedding-level metrics only (no clustering)')
    print(f'{"n_comp":<8} {"n_neigh":<8} {"metric":<13} {"m_dist":<7} {"trust":<8} {"recon_err":<10} {"nn_overlap"}')
    print('-' * 70)

    for _nc, _nn, _um, _md in _product(_nc_list, _nn_list, _um_list, _md_list):
        try:
            _red = umap.UMAP(n_components=_nc, n_neighbors=_nn, min_dist=_md,
                             metric=_um, random_state=42, low_memory=False)
            _emb = _red.fit_transform(X_weighted)

            # Trustworthiness (subsampled)
            _trust = _tw(_sub_X, _emb[_sub_idx], n_neighbors=min(15, _sub_n - 1))

            # Reconstruction error (UMAP's embedding loss via graph edge distances)
            # Approximate: mean pairwise distance distortion on subsample
            from scipy.spatial.distance import pdist, squareform
            _sub_emb = _emb[_sub_idx]
            # Use a smaller subsample for pairwise distances (expensive)
            _pdist_n = min(500, _sub_n)
            _pdist_idx = np.random.RandomState(42).choice(_sub_n, _pdist_n, replace=False)
            _d_orig = pdist(_sub_X[_pdist_idx], metric='correlation')
            _d_emb  = pdist(_sub_emb[_pdist_idx], metric='euclidean')
            # Normalise both to [0,1] range for comparable error
            _d_orig_n = _d_orig / (_d_orig.max() + 1e-10)
            _d_emb_n  = _d_emb / (_d_emb.max() + 1e-10)
            _recon_err = float(np.mean(np.abs(_d_orig_n - _d_emb_n)))

            # Neighbourhood overlap: fraction of k-NN preserved in embedding
            _nn_emb = _NN(n_neighbors=_k_overlap + 1).fit(_sub_emb)
            _emb_neighbors = _nn_emb.kneighbors(_sub_emb, return_distance=False)[:, 1:]
            _overlaps = [len(set(a) & set(b)) / _k_overlap
                         for a, b in zip(_orig_neighbors, _emb_neighbors)]
            _nn_ov = float(np.mean(_overlaps))

            umap_sweep_results.append({
                'n_components': _nc, 'n_neighbors': _nn, 'metric': _um, 'min_dist': _md,
                'trustworthiness': round(_trust, 3),
                'recon_error': round(_recon_err, 4),
                'nn_overlap': round(_nn_ov, 3)
            })
            print(f'{_nc:<8} {_nn:<8} {_um:<13} {_md:<7} {_trust:<8.3f} {_recon_err:<10.4f} {_nn_ov:.3f}')
        except Exception as _e:
            print(f'{_nc:<8} {_nn:<8} {_um:<13} {_md:<7} ERROR: {str(_e)[:60]}')

    umap_sweep_df = pd.DataFrame(umap_sweep_results)
    print(f'\nTop 15 by trustworthiness:')
    print(umap_sweep_df.sort_values('trustworthiness', ascending=False).head(15).to_string(index=False))

    print(f'\nPer-metric averages:')
    _ms = umap_sweep_df.groupby('metric').agg(
        avg_trust=('trustworthiness', 'mean'), best_trust=('trustworthiness', 'max'),
        avg_recon=('recon_error', 'mean'),
        avg_overlap=('nn_overlap', 'mean'), best_overlap=('nn_overlap', 'max')
    ).round(3)
    print(_ms.to_string())

    print(f'\nPer-min_dist averages:')
    _mds = umap_sweep_df.groupby('min_dist').agg(
        avg_trust=('trustworthiness', 'mean'),
        avg_recon=('recon_error', 'mean'),
        avg_overlap=('nn_overlap', 'mean')
    ).round(3)
    print(_mds.to_string())
else:
    print('UMAP sweep: SKIPPED (RUN_UMAP_SWEEP = False)')

In [42]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  UMAP SWEEP VISUALISATION — plotly dashboard of sweep results              ║
# ║  Inputs:  umap_sweep_df (from sweep cell above)                           ║
# ║  Outputs: Plotly interactive figure (inline display only)                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

try:
    umap_sweep_df
except NameError:
    print('No UMAP sweep data — set RUN_UMAP_SWEEP = True')
else:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    _metrics = sorted(umap_sweep_df['metric'].unique())
    _mc = {'correlation': '#E74C3C', 'cosine': '#3498DB', 'euclidean': '#2ECC71'}

    fig = make_subplots(rows=2, cols=3, subplot_titles=(
        'Trustworthiness by Metric', 'Recon Error by Metric', 'NN Overlap by Metric',
        'Trust vs n_components', 'Trust vs min_dist',
        'Best Config per Metric (Trust)'))

    for m in _metrics:
        s = umap_sweep_df[umap_sweep_df['metric'] == m]
        c = _mc.get(m, '#888')
        fig.add_trace(go.Histogram(x=s['trustworthiness'], name=m, marker_color=c, opacity=0.7), row=1, col=1)
        fig.add_trace(go.Histogram(x=s['recon_error'], name=m, marker_color=c, opacity=0.7,
                                   showlegend=False), row=1, col=2)
        fig.add_trace(go.Histogram(x=s['nn_overlap'], name=m, marker_color=c, opacity=0.7,
                                   showlegend=False), row=1, col=3)
        by_nc = s.groupby('n_components').agg(trust=('trustworthiness', 'mean')).reset_index()
        fig.add_trace(go.Scatter(x=by_nc['n_components'], y=by_nc['trust'], mode='markers+lines',
                                 name=m, marker=dict(size=8, color=c), showlegend=False), row=2, col=1)
        by_md = s.groupby('min_dist').agg(trust=('trustworthiness', 'mean')).reset_index()
        fig.add_trace(go.Scatter(x=by_md['min_dist'], y=by_md['trust'], mode='markers+lines',
                                 name=m, marker=dict(size=8, color=c), showlegend=False), row=2, col=2)

    best = umap_sweep_df.loc[umap_sweep_df.groupby('metric')['trustworthiness'].idxmax()]
    fig.add_trace(go.Bar(x=best['metric'], y=best['trustworthiness'],
                         marker_color=[_mc.get(m, '#888') for m in best['metric']],
                         text=[f'nc={r.n_components} nn={r.n_neighbors} md={r.min_dist}'
                               for _, r in best.iterrows()],
                         textposition='auto', showlegend=False), row=2, col=3)

    fig.update_xaxes(title_text='Trustworthiness', row=1, col=1)
    fig.update_xaxes(title_text='Recon Error', row=1, col=2)
    fig.update_xaxes(title_text='NN Overlap', row=1, col=3)
    fig.update_xaxes(title_text='n_components', row=2, col=1)
    fig.update_yaxes(title_text='Avg Trust', row=2, col=1)
    fig.update_xaxes(title_text='min_dist', row=2, col=2)
    fig.update_yaxes(title_text='Avg Trust', row=2, col=2)
    fig.update_yaxes(title_text='Best Trust', row=2, col=3)
    fig.update_layout(height=700, template='plotly_dark', title_text='UMAP Sweep — Embedding Quality')
    fig.show()

## 8 — HDBSCAN Density-Based Clustering

### Why Not K-Means?

The classic clustering algorithm K-Means requires you to specify the number of clusters in advance. For a streaming service with unknown category structure, that's a problem — we don't know how many "rails" the data naturally contains. K-Means also forces every point into a cluster, even if it doesn't fit anywhere, and assumes clusters are roughly spherical and equally sized.

### How HDBSCAN Works (Intuition)

HDBSCAN (Hierarchical Density-Based Spatial Clustering of Applications with Noise) works differently:

1. **Build a density landscape** — Imagine the 20D UMAP space as a terrain where elevation represents point density. Mountains are dense clusters; valleys are sparse boundaries.

2. **Find the mountains** — HDBSCAN identifies connected components of the density landscape above varying thresholds, building a hierarchy of clusters at different density levels.

3. **Select the most persistent clusters** — Using the Excess of Mass (EOM) method, it selects clusters that persist across the widest range of density thresholds — these are the "real" groupings, not noise artifacts.

4. **Label outliers** — Points in low-density valleys between mountains are labeled as outliers (-1). These are films that genuinely don't fit any cluster, and forcing them into one would reduce cluster quality.

### Key Parameters

- **min_cluster_size (48):** The smallest group of points that can be considered a cluster. Lower values find more (and smaller) clusters; higher values enforce a minimum rail size. 48 was a DBCV sweet spot — going to 45 caused a quality cliff.
- **min_samples (10):** How dense a region must be to count as a "core" point. Higher values make the algorithm more conservative about what counts as a cluster.
- **cluster_selection_epsilon (0.2):** Merges clusters that are closer than this distance, preventing over-fragmentation of groups that are very close together.
- **selection_method ('eom'):** Excess of Mass selection favors larger, more persistent clusters over highly granular splits.

### Quality Metrics

- **Silhouette score:** How similar each point is to its own cluster compared to the nearest other cluster. Ranges from -1 (wrong cluster) to +1 (perfect separation).
- **DBCV (Density-Based Cluster Validity):** The gold standard for density-based clustering. Measures whether clusters are internally dense and separated by low-density regions. Values above 0.4 indicate good structure.

**Inputs:** `embedding_cluster` (20D UMAP embedding), HDBSCAN hyperparameters from config.

**Process:** Fit HDBSCAN with `prediction_data=True` (needed for soft clustering later) and `gen_min_span_tree=True` (needed for DBCV). Compute cluster labels, membership probabilities, silhouette, and DBCV.

**Outputs:** `clusterer` (fitted HDBSCAN object), `cluster_labels` (array, -1 = outlier), `probabilities` (membership confidence), `n_clusters`, `n_outliers`, `n_total`, `sil`, `dbcv_score`.

<div style="display:flex; align-items:center; justify-content:center; gap:16px; padding:18px; background:linear-gradient(135deg, #ffebee 0%, #ffcdd2 100%); border-radius:12px; margin:10px 0;">
  <div style="text-align:center; padding:14px 20px; background:white; border-radius:8px; border:2px solid #F44336;">
    <b style="font-size:15px;">20D UMAP Space</b><br>
    <small style="color:#666;">Movies form natural<br>dense regions</small>
  </div>
  <div style="text-align:center;">
    <div style="font-size:28px; color:#F44336;">→ HDBSCAN →</div>
    <small style="color:#C62828;">Find density<br>peaks</small>
  </div>
  <div style="text-align:center; padding:14px 20px; background:white; border-radius:8px; border:2px solid #F44336;">
    <b style="font-size:15px;">Clusters + Outliers</b><br>
    <small style="color:#666;">Each cluster = a "rail"<br>~7% outliers = truly<br>uncategorisable</small>
  </div>
</div>

> **Why HDBSCAN?** Unlike K-means, HDBSCAN doesn't need a pre-set number of clusters — it discovers them from the data's density landscape. Dense regions become clusters; sparse regions between them become natural boundaries. Films that don't belong anywhere are flagged as outliers rather than forced into a bad fit.

In [43]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HDBSCAN CLUSTERING — density-based cluster discovery on 20D UMAP         ║
# ║  Inputs:  embedding_cluster (n_movies × 20), HDBSCAN config params        ║
# ║  Outputs: clusterer, cluster_labels, probabilities, n_clusters,           ║
# ║           n_outliers, n_total, sil, dbcv_score                            ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

from sklearn.metrics import silhouette_score

# ── Fit HDBSCAN ──────────────────────────────────────────────────────────────
# HDBSCAN finds clusters as connected components of the density-based
# mutual reachability graph. Points in low-density regions become outliers (-1).
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,   # 48 — smallest valid cluster
    min_samples=HDBSCAN_MIN_SAMPLES,             # 10 — core point density threshold
    cluster_selection_epsilon=HDBSCAN_EPSILON,    # 0.2 — merge clusters within this distance
    metric='euclidean',                           # Distance metric on 20D UMAP space
    cluster_selection_method=HDBSCAN_SELECTION_METHOD,  # 'eom' — Excess of Mass
    prediction_data=True,                         # Needed for approximate_predict (soft clustering)
    gen_min_span_tree=True                        # Needed for DBCV (relative_validity_)
)

# fit_predict returns cluster label per point (-1 = outlier/noise)
cluster_labels = clusterer.fit_predict(embedding_cluster)
# Membership probability: how confidently each point belongs to its cluster (0-1)
probabilities = clusterer.probabilities_

# ── Compute summary statistics ────────────────────────────────────────────────
n_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)  # Exclude outlier label
n_outliers = int(np.sum(cluster_labels == -1))    # Count of noise points
n_total = len(cluster_labels)                     # Total movie count

# Silhouette score: measures how similar each point is to its own cluster vs nearest cluster
# Only computed on clustered (non-outlier) points, subsampled for speed
sil = silhouette_score(
    embedding_cluster[cluster_labels != -1],       # Exclude outliers
    cluster_labels[cluster_labels != -1],
    sample_size=min(2000, int((cluster_labels != -1).sum())),
    random_state=42
) if n_clusters >= 2 else -1

# DBCV: density-based cluster validity — the gold standard metric for HDBSCAN
# Measures whether clusters are internally dense and externally separated
dbcv_score = clusterer.relative_validity_

print(f'HDBSCAN complete')
print(f'  Clusters:   {n_clusters}')
print(f'  Outliers:   {n_outliers} ({100 * n_outliers / n_total:.1f}%)')
print(f'  Silhouette: {sil:.3f}')
print(f'  DBCV:       {dbcv_score:.3f}')

HDBSCAN complete
  Clusters:   43
  Outliers:   589 (7.4%)
  Silhouette: 0.609
  DBCV:       0.415


### 8-viz — Clustering Results

Three views of the HDBSCAN output that together tell you whether the clustering "worked":

**Panel 1 — 3D Cluster Scatter:** Each colour is one rail; grey points are outliers. Well-separated colour blobs = distinct clusters. If colours heavily overlap in the 3D view, remember the actual clustering was done in 20D where there's more room for separation.

**Panel 2 — Cluster Size Distribution:** A horizontal bar chart showing how many movies are in each cluster. Healthy clustering has a mix of sizes. If one cluster dominates (e.g., 3,000 films) while others have only 50, the model may be finding one big "everything else" bucket rather than meaningful distinctions.

**Panel 3 — Membership Probability Distribution:** HDBSCAN assigns each clustered point a probability (0–1) reflecting how confidently it belongs to its cluster. A sharp peak near 1.0 means most movies are confidently placed. A flat or left-skewed distribution suggests many borderline assignments.

**Inputs:** `embedding_3d`, `cluster_labels`, `probabilities`, `n_clusters`, `n_outliers`.

**Process:** Three matplotlib subplots: 3D scatter (outliers rendered behind in grey), horizontal bar chart of cluster sizes with median line, and histogram of membership probabilities for clustered films.

**Outputs:** Matplotlib figure (displayed inline). No new variables created.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HDBSCAN VISUALISATION — 3D scatter, size distribution, confidence dist   ║
# ║  Inputs:  embedding_3d, cluster_labels, probabilities, n_clusters         ║
# ║  Outputs: Matplotlib 3-panel figure (inline display only)                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Visualisation: HDBSCAN Clustering Results ────────────────────────────────
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.cm as cm

_fig = plt.figure(figsize=(20, 6))

# ── Panel 1: 3D UMAP colored by cluster ─────────────────────────────────────
_ax1 = _fig.add_subplot(131, projection='3d')

_unique_clusters = sorted(set(cluster_labels) - {-1})
_cmap = cm.get_cmap('tab20', len(_unique_clusters))
_cluster_to_color = {cid: _cmap(i) for i, cid in enumerate(_unique_clusters)}
_cluster_to_color[-1] = (0.75, 0.75, 0.75, 0.15)

_point_colors = [_cluster_to_color.get(l, (0.5, 0.5, 0.5, 0.2)) for l in cluster_labels]

# Plot outliers first (so they're behind)
_outlier_mask = cluster_labels == -1
_cluster_mask = ~_outlier_mask

_ax1.scatter(
    embedding_3d[_outlier_mask, 0], embedding_3d[_outlier_mask, 1], embedding_3d[_outlier_mask, 2],
    s=2, alpha=0.08, c='grey', label=f'Outliers ({_outlier_mask.sum():,})'
)
_ax1.scatter(
    embedding_3d[_cluster_mask, 0], embedding_3d[_cluster_mask, 1], embedding_3d[_cluster_mask, 2],
    s=5, alpha=0.5, c=[_cluster_to_color[l] for l in cluster_labels[_cluster_mask]],
    edgecolors='none'
)

_ax1.set_title(f'{n_clusters} Clusters Discovered', fontsize=13, fontweight='bold', pad=12)
_ax1.set_xlabel('UMAP-1', fontsize=9)
_ax1.set_ylabel('UMAP-2', fontsize=9)
_ax1.set_zlabel('UMAP-3', fontsize=9)
_ax1.tick_params(labelsize=7)
_ax1.view_init(elev=25, azim=135)

# ── Panel 2: Cluster Size Distribution ───────────────────────────────────────
_ax2 = _fig.add_subplot(132)

_cluster_sizes = pd.Series(cluster_labels[cluster_labels >= 0]).value_counts().sort_values(ascending=True)
_size_colors = [_cluster_to_color[cid] for cid in _cluster_sizes.index]

_ax2.barh(range(len(_cluster_sizes)), _cluster_sizes.values, color=_size_colors, edgecolor='white', linewidth=0.3)
_ax2.set_yticks(range(len(_cluster_sizes)))
_ax2.set_yticklabels([f'C{cid}' for cid in _cluster_sizes.index], fontsize=7)
_ax2.set_xlabel('Number of Movies', fontsize=12)
_ax2.set_title(f'Cluster Sizes ({n_clusters} clusters)', fontsize=13, fontweight='bold', pad=12)
_ax2.spines[['top', 'right']].set_visible(False)

# Annotate median
_median_size = int(np.median(_cluster_sizes.values))
_ax2.axvline(_median_size, color='#F44336', linestyle='--', linewidth=1.5, alpha=0.7,
             label=f'Median: {_median_size}')
_ax2.legend(fontsize=10)

# ── Panel 3: Membership Probability Distribution ────────────────────────────
_ax3 = _fig.add_subplot(133)

_ax3.hist(probabilities[cluster_labels >= 0], bins=50, color='#F44336', edgecolor='white', alpha=0.85,
          label='Clustered films')
_med_prob = np.median(probabilities[cluster_labels >= 0])
_ax3.axvline(_med_prob, color='#1a1a2e', linestyle='--', linewidth=2,
             label=f'Median: {_med_prob:.2f}')
_ax3.set_xlabel('Membership Probability', fontsize=12)
_ax3.set_ylabel('Number of Films', fontsize=12)
_ax3.set_title('Cluster Confidence Distribution', fontsize=13, fontweight='bold', pad=12)
_ax3.spines[['top', 'right']].set_visible(False)
_ax3.legend(fontsize=10)

_fig.tight_layout()
plt.show()

# Print summary
print(f'\n  Total: {n_total:,} movies → {n_clusters} clusters + {n_outliers} outliers ({100*n_outliers/n_total:.1f}%)')
print(f'  Silhouette: {sil:.3f}  |  DBCV: {dbcv_score:.3f}')
print(f'  Cluster sizes: {_cluster_sizes.values.min()} – {_cluster_sizes.values.max()} (median {_median_size})')
print(f'  Avg membership prob: {probabilities[cluster_labels >= 0].mean():.3f}')

### 8.1 — HDBSCAN Parameter Sweep

Like the UMAP sweep, this is an exhaustive grid search — but now over HDBSCAN's clustering parameters. The sweep tests ~350 combinations and measures four quality metrics per configuration.

### What Each Parameter Controls

- **min_cluster_size** [5, 10, 15, 20, 30, 40, 60]: The minimum number of films required to form a rail. This is the most impactful parameter — it directly controls granularity vs. coverage.
- **min_samples** [1, 3, 5, 10, 15]: How dense a region must be for its core points. Higher values = more conservative clustering.
- **epsilon** [0.0, 0.1, 0.2, 0.3, 0.5]: Distance threshold for merging nearby clusters. Prevents over-fragmentation.
- **selection_method** ['eom', 'leaf']: EOM (Excess of Mass) favors large persistent clusters; Leaf favors many small homogeneous ones.

### The Persistence Metric

Beyond silhouette and DBCV, we measure **persistence** — how stable the cluster labels are when you slightly change the parameters. For each configuration, we compute the Adjusted Rand Index (ARI) against its 10 nearest neighbours in parameter space. High persistence means the result isn't fragile; small parameter changes don't radically reshape the clusters.

**Inputs:** `embedding_cluster` (20D UMAP), `RUN_HDBSCAN_SWEEP` toggle.

**Process:** For each of ~350 combinations, fit HDBSCAN, compute silhouette, DBCV, avg probability, and persistence (pairwise ARI). Print top-10 tables by DBCV and by persistence, plus EOM vs Leaf comparison.

**Outputs:** `hdb_sweep_results`, `hdb_sweep_df` (DataFrame). Skipped if `RUN_HDBSCAN_SWEEP = False`.

In [44]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HDBSCAN PARAMETER SWEEP — grid search for optimal clustering config      ║
# ║  Inputs:  embedding_cluster (20D UMAP), RUN_HDBSCAN_SWEEP toggle          ║
# ║  Outputs: hdb_sweep_results (list), hdb_sweep_df (DataFrame)              ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

if RUN_HDBSCAN_SWEEP:
    from itertools import product as _product
    from sklearn.metrics import silhouette_score as _sil

    _mcs_list = [5, 10, 15, 20, 30, 40, 60]
    _ms_list  = [1, 3, 5, 10, 15]
    _eps_list = [0.0, 0.1, 0.2, 0.3, 0.5]
    _sel_list = ['eom', 'leaf']

    hdb_sweep_results = []
    _total = len(_mcs_list) * len(_ms_list) * len(_eps_list) * len(_sel_list)
    print(f'HDBSCAN sweep: {_total} combos')
    print(f'{"mcs":<6} {"ms":<5} {"eps":<6} {"sel":<6} {"clust":<8} {"out%":<8} {"sil":<8} {"DBCV":<8} {"avg_prob"}')
    print('-' * 70)

    _all_labels = []

    for _mcs, _ms, _eps, _sel in _product(_mcs_list, _ms_list, _eps_list, _sel_list):
        _cl = hdbscan.HDBSCAN(min_cluster_size=_mcs, min_samples=_ms,
                               cluster_selection_epsilon=_eps, metric='euclidean',
                               cluster_selection_method=_sel,
                               gen_min_span_tree=True, prediction_data=True)
        _lbl = _cl.fit_predict(embedding_cluster)
        _n_c = len(set(_lbl)) - (1 if -1 in _lbl else 0)
        _n_o = int(np.sum(_lbl == -1))
        _op = _n_o / len(_lbl)

        if _n_c >= 2:
            _v = _lbl != -1
            _s = _sil(embedding_cluster[_v], _lbl[_v],
                       sample_size=min(2000, int(_v.sum())), random_state=42)
            _dbcv = _cl.relative_validity_
        else:
            _s, _dbcv = -1.0, -1.0

        _avg_prob = float(_cl.probabilities_[_lbl != -1].mean()) if _n_c >= 1 else 0.0
        _all_labels.append(_lbl.copy())

        hdb_sweep_results.append({
            'min_cluster_size': _mcs, 'min_samples': _ms, 'epsilon': _eps,
            'selection_method': _sel,
            'n_clusters': _n_c, 'outlier_pct': round(_op, 3),
            'silhouette': round(_s, 3), 'dbcv': round(_dbcv, 3),
            'avg_probability': round(_avg_prob, 3)})

    hdb_sweep_df = pd.DataFrame(hdb_sweep_results)

    # Cluster persistence: pairwise ARI between neighbouring runs
    from sklearn.metrics import adjusted_rand_score as _ari
    _n_runs = len(_all_labels)
    _ari_per_run = []
    for i in range(_n_runs):
        _aris = []
        for j in range(max(0, i-5), min(_n_runs, i+6)):
            if i != j:
                _aris.append(_ari(_all_labels[i], _all_labels[j]))
        _ari_per_run.append(round(np.mean(_aris), 3) if _aris else 0.0)
    hdb_sweep_df['persistence'] = _ari_per_run

    print(f'\nTop 10 by DBCV:')
    print(hdb_sweep_df.sort_values('dbcv', ascending=False).head(10)[
        ['min_cluster_size', 'min_samples', 'epsilon', 'selection_method',
         'n_clusters', 'dbcv', 'silhouette', 'persistence', 'avg_probability']
    ].to_string(index=False))

    print(f'\nTop 10 by persistence:')
    print(hdb_sweep_df.sort_values('persistence', ascending=False).head(10)[
        ['min_cluster_size', 'min_samples', 'epsilon', 'selection_method',
         'n_clusters', 'dbcv', 'silhouette', 'persistence', 'avg_probability']
    ].to_string(index=False))

    print(f'\nEOM vs Leaf summary:')
    _sel_summary = hdb_sweep_df.groupby('selection_method').agg(
        avg_sil=('silhouette', 'mean'), best_sil=('silhouette', 'max'),
        avg_dbcv=('dbcv', 'mean'), best_dbcv=('dbcv', 'max'),
        avg_clusters=('n_clusters', 'mean'),
        avg_persistence=('persistence', 'mean'),
        avg_outlier=('outlier_pct', 'mean')
    ).round(3)
    print(_sel_summary.to_string())

    print(f'\nTotal combos: {len(hdb_sweep_df)}')
else:
    print('HDBSCAN sweep: SKIPPED (RUN_HDBSCAN_SWEEP = False)')

HDBSCAN sweep: 350 combos
mcs    ms    eps    sel    clust    out%     sil      DBCV     avg_prob
----------------------------------------------------------------------


KeyboardInterrupt: 

In [45]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  HDBSCAN SWEEP VISUALISATION — plotly dashboard of sweep results           ║
# ║  Inputs:  hdb_sweep_df (from sweep cell above)                            ║
# ║  Outputs: Plotly interactive figure (inline display only)                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

try:
    hdb_sweep_df
except NameError:
    print('No HDBSCAN sweep data — set RUN_HDBSCAN_SWEEP = True')
else:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    _sc = {'eom': '#E74C3C', 'leaf': '#3498DB'}

    fig = make_subplots(rows=2, cols=3, subplot_titles=(
        'DBCV: EOM vs Leaf', 'Silhouette: EOM vs Leaf', 'Persistence: EOM vs Leaf',
        'DBCV vs Silhouette', 'DBCV vs Persistence', 'Clusters vs Avg Probability'),
        horizontal_spacing=0.08, vertical_spacing=0.15)

    for sel in ['eom', 'leaf']:
        s = hdb_sweep_df[hdb_sweep_df['selection_method'] == sel]
        c = _sc[sel]
        fig.add_trace(go.Histogram(x=s['dbcv'], name=sel, marker_color=c, opacity=0.7), row=1, col=1)
        fig.add_trace(go.Histogram(x=s['silhouette'], name=sel, marker_color=c, opacity=0.7,
                                   showlegend=False), row=1, col=2)
        fig.add_trace(go.Histogram(x=s['persistence'], name=sel, marker_color=c, opacity=0.7,
                                   showlegend=False), row=1, col=3)

    fig.add_trace(go.Scatter(
        x=hdb_sweep_df['silhouette'], y=hdb_sweep_df['dbcv'], mode='markers',
        marker=dict(size=5,
                    color=[_sc.get(m, '#888') for m in hdb_sweep_df['selection_method']],
                    opacity=0.6),
        showlegend=False), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=hdb_sweep_df['persistence'], y=hdb_sweep_df['dbcv'], mode='markers',
        marker=dict(size=5, color=hdb_sweep_df['outlier_pct']*100, colorscale='Inferno',
                    showscale=True, colorbar=dict(title='Out%', len=0.35, y=0.18,
                                                  x=0.62, thickness=12)),
        showlegend=False), row=2, col=2)

    fig.add_trace(go.Scatter(
        x=hdb_sweep_df['n_clusters'], y=hdb_sweep_df['avg_probability'], mode='markers',
        marker=dict(size=5,
                    color=[_sc.get(m, '#888') for m in hdb_sweep_df['selection_method']],
                    opacity=0.6),
        showlegend=False), row=2, col=3)

    fig.update_xaxes(title_text='DBCV', row=1, col=1)
    fig.update_xaxes(title_text='Silhouette', row=1, col=2)
    fig.update_xaxes(title_text='Persistence (ARI)', row=1, col=3)
    fig.update_xaxes(title_text='Silhouette', row=2, col=1)
    fig.update_yaxes(title_text='DBCV', row=2, col=1)
    fig.update_xaxes(title_text='Persistence', row=2, col=2)
    fig.update_yaxes(title_text='DBCV', row=2, col=2)
    fig.update_xaxes(title_text='n_clusters', row=2, col=3)
    fig.update_yaxes(title_text='Avg Probability', row=2, col=3)
    fig.update_layout(
        height=700, template='plotly_dark',
        title_text='HDBSCAN Sweep — EOM vs Leaf + Quality',
        legend=dict(orientation='h', yanchor='bottom', y=1.02,
                    xanchor='right', x=1))
    fig.show()

## 8.5 — XGBoost Cluster Validation & Interpretation

### The Validation Problem

HDBSCAN found our clusters using density in a 20D UMAP embedding. But how do we know these clusters are *real patterns* in the data and not just artifacts of the specific UMAP projection? If we ran UMAP with a different random seed, would we get completely different clusters?

The answer is to **validate with an independent method**. We train XGBoost — a gradient-boosted decision tree classifier — to predict cluster membership directly from the **original IDF-weighted features** (not the UMAP embedding). If XGBoost achieves high cross-validated accuracy, it means the cluster boundaries exist in the original feature space, not just in the UMAP projection.

### What This Section Produces

**Cross-validated accuracy** — 5-fold CV accuracy tells us how learnable the clusters are. Above 90% = highly learnable with clear boundaries. 75-90% = moderate overlap. Below 75% = clusters may be fuzzy or arbitrary.

**A trained XGBoost model** — Used downstream for two things: SHAP feature importance (Section 8.5b) and outlier recovery (Section 8.5c).

**Labeled outlier data** — The clustered/outlier split of `X_weighted` is prepared here for use in outlier recovery.

**Inputs:** `X_weighted` (IDF-weighted features), `cluster_labels`, `feature_names`.

**Process:**
1. Split into clustered films (`cluster_labels != -1`) and outliers (`cluster_labels == -1`).
2. Encode cluster labels to consecutive integers for XGBoost.
3. Run 5-fold cross-validation to measure accuracy.
4. Train final model on all clustered data for SHAP and outlier recovery.

**Outputs:** `xgb_clf` (trained model), `cv_scores`, `X_clustered`, `X_outliers`, `y_encoded`, `le` (LabelEncoder), `outlier_indices`.

<div style="display:flex; align-items:center; justify-content:center; gap:16px; padding:18px; background:linear-gradient(135deg, #e0f7fa 0%, #b2ebf2 100%); border-radius:12px; margin:10px 0;">
  <div style="text-align:center; padding:14px 20px; background:white; border-radius:8px; border:2px solid #00BCD4;">
    <b style="font-size:15px;">HDBSCAN Labels</b><br>
    <small style="color:#666;">Unsupervised clusters</small>
  </div>
  <div style="text-align:center;">
    <div style="font-size:28px; color:#00BCD4;">→ XGBoost →</div>
    <small style="color:#00838F;">Can a supervised model<br>learn these labels?</small>
  </div>
  <div style="text-align:center; padding:14px 20px; background:white; border-radius:8px; border:2px solid #00BCD4;">
    <b style="font-size:15px;">High CV Accuracy?</b><br>
    <small style="color:#666;">✓ Clusters are real<br>✗ Clusters are noise</small>
  </div>
</div>

> **The validation logic:** If an *independent supervised model* (XGBoost) can predict which cluster a movie belongs to with high accuracy from the original features, then those clusters have clear, learnable boundaries — they're real patterns, not statistical artifacts.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  XGBOOST CLUSTER VALIDATION — supervised test of unsupervised clusters    ║
# ║  Inputs:  X_weighted, cluster_labels, feature_names                       ║
# ║  Outputs: xgb_clf (trained model), cv_scores, X_clustered, X_outliers,   ║
# ║           y_encoded, le (LabelEncoder), outlier_indices                   ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import xgboost as xgb
import shap
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder

# ══════════════════════════════════════════════════════════════════════════════
# STEP 1: Train XGBoost on clustered films (exclude outliers)
# ══════════════════════════════════════════════════════════════════════════════
# NOTE: XGBoost uses X_weighted (IDF-weighted 283 features).
# Same feature space as UMAP input, preserving interpretable names for SHAP.

# Separate clustered films from outliers
clustered_mask = cluster_labels != -1
outlier_mask = cluster_labels == -1

X_clustered = X_weighted[clustered_mask]
y_clustered = cluster_labels[clustered_mask]

X_outliers = X_weighted[outlier_mask]
outlier_indices = np.where(outlier_mask)[0]

# Encode labels to consecutive integers for XGBoost
le = LabelEncoder()
y_encoded = le.fit_transform(y_clustered)

print(f'Training data:  {X_clustered.shape[0]:,} films in {len(le.classes_)} clusters')
print(f'Outlier data:   {X_outliers.shape[0]:,} films to recover')
print(f'Feature space:  {X_clustered.shape[1]} IDF-weighted features (original names preserved)')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 2: Cross-validated accuracy (cluster validation)
# ══════════════════════════════════════════════════════════════════════════════

xgb_clf = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

cv_scores = cross_val_score(xgb_clf, X_clustered, y_encoded, cv=5, scoring='accuracy')

print(f'\n{"="*60}')
print(f'CLUSTER VALIDATION — 5-Fold Cross-Validated Accuracy')
print(f'{"="*60}')
print(f'  Fold scores:   {[f"{s:.3f}" for s in cv_scores]}')
print(f'  Mean accuracy: {cv_scores.mean():.1%} ± {cv_scores.std():.1%}')
print(f'{"="*60}')
if cv_scores.mean() >= 0.90:
    print(f'  ✓ Clusters are highly learnable — clear, consistent boundaries')
elif cv_scores.mean() >= 0.75:
    print(f'  ~ Clusters are moderately learnable — some overlap exists')
else:
    print(f'  ✗ Clusters are hard to learn — may be fuzzy or arbitrary')

# ══════════════════════════════════════════════════════════════════════════════
# STEP 3: Train final model on all clustered data for SHAP + outlier recovery
# ══════════════════════════════════════════════════════════════════════════════

xgb_clf.fit(X_clustered, y_encoded)
print(f'\nFinal model trained on all {X_clustered.shape[0]:,} clustered films')

### 8.5-viz — Cluster Validation Results

**How to read this:** The left panel shows the accuracy of each of the 5 cross-validation folds. Consistent bars (small spread) mean the result is robust. A single low fold would suggest one data partition was harder than others, possibly indicating some cluster overlap.

The right panel provides a text summary with a verdict. The key number is the **mean accuracy**: ≥90% means the clusters have clear, learnable boundaries in the original feature space — they aren't just UMAP artifacts. Below 75% would suggest the clusters are fuzzy or that UMAP created artificial separations.

**Why cross-validation matters:** Training accuracy would always be high (the model memorizes the data). CV splits the data into 5 parts, trains on 4, and tests on the held-out 1 — this measures how well the cluster structure generalizes to unseen films.

**Inputs:** `cv_scores` (5-fold accuracy array from XGBoost).

**Process:** Left panel: per-fold accuracy bar chart with mean line. Right panel: monospace text summary with colour-coded verdict.

**Outputs:** Matplotlib figure (displayed inline). No new variables created.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  XGBOOST CV VISUALISATION — per-fold accuracy + verdict summary            ║
# ║  Inputs:  cv_scores (5-element accuracy array)                             ║
# ║  Outputs: Matplotlib 2-panel figure (inline display only)                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Visualisation: XGBoost Cross-Validation ──────────────────────────────────
_fig, _axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Left: Per-fold accuracy bars ────────────────────────────────────────────
_fold_colors = ['#00BCD4' if s >= cv_scores.mean() else '#FF9800' for s in cv_scores]
_bars = _axes[0].bar(range(1, len(cv_scores) + 1), cv_scores, color=_fold_colors,
                     edgecolor='white', linewidth=1.5, width=0.65)
_axes[0].axhline(cv_scores.mean(), color='#F44336', linestyle='--', linewidth=2,
                 label=f'Mean: {cv_scores.mean():.1%} ± {cv_scores.std():.1%}')
_axes[0].set_xlabel('Fold', fontsize=13)
_axes[0].set_ylabel('Accuracy', fontsize=13)
_axes[0].set_title('5-Fold Cross-Validation Accuracy', fontsize=14, fontweight='bold', pad=12)
_axes[0].set_ylim(0, 1.05)
_axes[0].legend(fontsize=12, loc='lower right')
_axes[0].spines[['top', 'right']].set_visible(False)

for _bar, _s in zip(_bars, cv_scores):
    _axes[0].text(_bar.get_x() + _bar.get_width()/2, _bar.get_height() + 0.01,
                  f'{_s:.1%}', ha='center', fontsize=11, fontweight='bold')

# ── Right: Interpretation summary ────────────────────────────────────────────
_axes[1].axis('off')

_verdict = '✓ HIGHLY LEARNABLE' if cv_scores.mean() >= 0.90 else ('~ MODERATELY LEARNABLE' if cv_scores.mean() >= 0.75 else '✗ WEAK')
_verdict_color = '#4CAF50' if cv_scores.mean() >= 0.90 else ('#FF9800' if cv_scores.mean() >= 0.75 else '#F44336')

_summary_text = f"""Cluster Validation Summary
{'━' * 35}

Mean CV Accuracy:  {cv_scores.mean():.1%}
Standard Dev:      ±{cv_scores.std():.1%}
Folds:             5

Verdict: {_verdict}

An independent supervised model
can predict HDBSCAN cluster labels
with {cv_scores.mean():.0%} accuracy from the original
282 features — confirming the
clusters have clear, learnable
boundaries."""

_axes[1].text(0.1, 0.95, _summary_text, transform=_axes[1].transAxes,
              fontsize=13, fontfamily='monospace', verticalalignment='top',
              bbox=dict(boxstyle='round,pad=0.8', facecolor=_verdict_color, alpha=0.15))

_fig.tight_layout()
plt.show()

### 8.5b — SHAP Feature Importance per Cluster

### What is SHAP?

SHAP (SHapley Additive exPlanations) comes from game theory. Imagine each feature as a "player" in a cooperative game where the goal is to correctly predict a cluster label. SHAP values measure each player's *marginal contribution* — how much does adding this feature improve the prediction, averaged over all possible combinations of other features?

This captures something simpler methods miss: **feature interactions**. If "heist" and "ensemble cast" *together* define a cluster but neither alone does, simple mean-comparison methods would miss both. SHAP correctly attributes importance to both features because they consistently improve predictions when combined.

### How We Use It

For each cluster, we compute the mean absolute SHAP value per feature across a subsample of films. Features with high mean |SHAP| are the ones XGBoost relies on most to identify that cluster. The top 10 features per cluster become the cluster's **SHAP profile** — a fingerprint of what makes it distinctive.

These profiles serve two purposes:
1. **Human understanding** — You can look at a cluster's top features and immediately understand what kind of movies it contains.
2. **LLM naming** — The SHAP profiles are fed to the LLM in Section 9 to generate precise, evocative rail names.

### Why TreeExplainer?

For tree-based models like XGBoost, SHAP's `TreeExplainer` computes **exact** SHAP values in polynomial time (not the exponential time that Shapley values normally require). This makes it practical to compute SHAP for 2,000 films × 283 features × 40+ clusters.

**Inputs:** `xgb_clf` (trained XGBoost model), `X_clustered` (IDF-weighted features for clustered films), `le` (label encoder), `feature_names`.

**Process:** Compute SHAP values using `TreeExplainer` on a 2,000-film subsample. For each cluster, extract the top 10 features by mean absolute SHAP value.

**Outputs:** `shap_values` / `shap_array` (raw SHAP values), `cluster_shap_features` (dict mapping `cluster_id → [(feature_name, mean_abs_shap), ...]`).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SHAP FEATURE IMPORTANCE — which features define each cluster?             ║
# ║  Inputs:  xgb_clf, X_clustered, le, feature_names                         ║
# ║  Outputs: shap_values, shap_array, cluster_shap_features (dict)           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ══════════════════════════════════════════════════════════════════════════════
# SHAP values for each cluster — which features define each rail?
# ══════════════════════════════════════════════════════════════════════════════

# Use TreeExplainer for fast exact SHAP on tree models
explainer = shap.TreeExplainer(xgb_clf)

# Compute SHAP values on a subsample for speed (2000 films is plenty)
_shap_n = min(2000, X_clustered.shape[0])
_shap_idx = np.random.RandomState(42).choice(X_clustered.shape[0], _shap_n, replace=False)
shap_values = explainer.shap_values(X_clustered[_shap_idx])

# shap_values shape: (n_samples, n_features, n_classes) or list of (n_samples, n_features)
# For multi-class XGBoost, shap_values is an array of shape (n_samples, n_features, n_classes)
if isinstance(shap_values, list):
    # Older SHAP returns list of arrays per class
    shap_array = np.array(shap_values)  # (n_classes, n_samples, n_features)
elif shap_values.ndim == 3:
    shap_array = shap_values.transpose(2, 0, 1)  # → (n_classes, n_samples, n_features)
else:
    shap_array = shap_values[np.newaxis, :, :]  # binary case

print(f'SHAP values computed for {_shap_n} films across {shap_array.shape[0]} classes')

# ── Extract top SHAP features per cluster ────────────────────────────────────
cluster_shap_features = {}  # {cluster_id: [(feature_name, mean_abs_shap), ...]}

print(f'\n{"Cluster":<10} {"Size":<8} {"Top 5 SHAP Features"}')
print("=" * 90)

for i, cid in enumerate(le.classes_):
    original_cid = le.inverse_transform([cid])[0]
    # Mean absolute SHAP value per feature for this class
    mean_abs_shap = np.abs(shap_array[i]).mean(axis=0)  # (n_features,)
    top_idx = np.argsort(mean_abs_shap)[::-1][:10]
    top_feats = [(feature_names[j], float(mean_abs_shap[j])) for j in top_idx]
    cluster_shap_features[original_cid] = top_feats

    n_films = int((cluster_labels == original_cid).sum())
    feat_str = ", ".join([f"{n}" for n, s in top_feats[:5]])
    print(f'{original_cid:<10} {n_films:<8} {feat_str}')

print(f'\nSHAP features extracted for {len(cluster_shap_features)} clusters')


### 8.5b-viz — SHAP Feature Importance Heatmap

**How to read this heatmap:** Each column is a cluster (rail), each row is a genome feature, and the cell colour intensity represents how important that feature is for identifying that cluster. Bright (hot) cells mean the feature is critical for distinguishing that rail from all others.

**What to look for:**
- **Bright columns with unique features** = well-defined rails with distinctive identities (e.g., one rail dominated by "claymation" + "animation" + "whimsical").
- **Features that light up across many columns** = features that are generally useful for clustering but don't define any single rail (e.g., "violence" might help distinguish multiple action-flavored rails).
- **Sparse columns** = rails that are defined by subtle combinations rather than standout single features — these are often the hardest for the LLM to name.

The heatmap shows the **top 30 most globally impactful features** — the features that matter most when summed across all clusters. This is a subset of the full 283; most features have minimal SHAP impact.

**Inputs:** `cluster_shap_features` (top SHAP features per cluster), `feature_names`.

**Process:** Build a (features × clusters) matrix of mean absolute SHAP values. Select top 30 features by total importance. Render with `YlOrRd` colormap.

**Outputs:** Matplotlib heatmap figure (displayed inline). No new variables created.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SHAP HEATMAP — feature importance across all clusters                     ║
# ║  Inputs:  cluster_shap_features, feature_names                             ║
# ║  Outputs: Matplotlib heatmap figure (inline display only)                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Visualisation: SHAP Feature Importance Heatmap ───────────────────────────
# Build a matrix of top SHAP features across all clusters

# Collect unique top features across all clusters
_all_top_feats = set()
for _cid, _feats in cluster_shap_features.items():
    for _fname, _score in _feats[:5]:
        _all_top_feats.add(_fname)

# Limit to top N most impactful features overall
_feat_total_importance = {}
for _cid, _feats in cluster_shap_features.items():
    for _fname, _score in _feats:
        _feat_total_importance[_fname] = _feat_total_importance.get(_fname, 0) + _score

_top_global_feats = sorted(_feat_total_importance.keys(),
                           key=lambda f: _feat_total_importance[f], reverse=True)[:30]

# Build heatmap matrix
_sorted_clusters = sorted(cluster_shap_features.keys())
_heatmap_data = np.zeros((len(_top_global_feats), len(_sorted_clusters)))

for _j, _cid in enumerate(_sorted_clusters):
    _feat_dict = dict(cluster_shap_features[_cid])
    for _i, _fname in enumerate(_top_global_feats):
        _heatmap_data[_i, _j] = _feat_dict.get(_fname, 0.0)

_fig, _ax = plt.subplots(figsize=(max(16, n_clusters * 0.45), 10))

_im = _ax.imshow(_heatmap_data, cmap='YlOrRd', aspect='auto', interpolation='nearest')

_ax.set_xticks(range(len(_sorted_clusters)))
_ax.set_xticklabels([f'C{c}' for c in _sorted_clusters], fontsize=7, rotation=90)
_ax.set_yticks(range(len(_top_global_feats)))
_ax.set_yticklabels(_top_global_feats, fontsize=9)

_ax.set_title('SHAP Feature Importance by Cluster — Top 30 Global Features',
              fontsize=15, fontweight='bold', pad=14)
_ax.set_xlabel('Cluster', fontsize=12)

_cbar = _fig.colorbar(_im, ax=_ax, shrink=0.8, label='Mean |SHAP value|')

_fig.tight_layout()
plt.show()

print(f'  Heatmap shows top 30 most impactful features across {len(_sorted_clusters)} clusters')
print(f'  Bright cells = features that DEFINE that cluster')

### 8.5c — Outlier Recovery via XGBoost Prediction

### The Outlier Problem

HDBSCAN labels ~7% of films as outliers (-1) — movies in low-density regions between clusters that don't confidently fit anywhere. For a streaming service, losing 7% of your catalog to an "uncategorised" bucket is unacceptable. But forcing them into clusters (like K-Means would) is worse — it pollutes rail quality.

### The Recovery Strategy

XGBoost, trained on the confidently-clustered films, can make predictions about outliers using `predict_proba`. For each outlier film, it outputs a probability distribution across all clusters. If the model is 85% confident that *The Truman Show* belongs to the "Philosophical Sci-Fi" rail, that's a strong recovery candidate. If it's only 20% confident about *Mulholland Drive*, that film is a genuine outlier — it doesn't cleanly fit any single rail.

### Confidence Tiers

- **≥50% confidence (recoverable):** Strong candidates for cluster assignment. These films are near the boundary of a cluster and likely belong there — HDBSCAN just wasn't quite confident enough.
- **30–50% confidence (borderline):** The model has a slight preference but isn't sure. These might be multi-genre films that straddle two rails.
- **<30% confidence (true outliers):** Genuinely uncategorisable films. These might be experimental works, genre mashups, or films with unusual profiles.

Recovery suggestions appear in the dashboard's **Outlier Review** page for curator approval — the model suggests, a human decides.

**Inputs:** `xgb_clf` (trained model), `X_outliers` (IDF-weighted features for outlier films), `outlier_indices`, `le` (label encoder), `tt_codes`, `title_lookup`.

**Process:** Run `predict_proba` on all outlier films, extract the best class and confidence, build the `recovery_suggestions` dict.

**Outputs:** `recovery_suggestions` (dict mapping `global_index → {suggested_cluster, confidence, tt_code, title}`), `RECOVERY_THRESHOLD` constant (0.5).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  OUTLIER RECOVERY — predict cluster membership for orphaned films          ║
# ║  Inputs:  xgb_clf, X_outliers, outlier_indices, le, tt_codes,            ║
# ║           title_lookup                                                     ║
# ║  Outputs: recovery_suggestions (dict), RECOVERY_THRESHOLD                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ══════════════════════════════════════════════════════════════════════════════
# Outlier Recovery — predict cluster membership for orphaned films
# ══════════════════════════════════════════════════════════════════════════════

RECOVERY_THRESHOLD = 0.5  # minimum confidence to suggest recovery

if X_outliers.shape[0] > 0:
    outlier_proba = xgb_clf.predict_proba(X_outliers)  # (n_outliers, n_classes)
    outlier_best_class = np.argmax(outlier_proba, axis=1)
    outlier_best_conf = np.max(outlier_proba, axis=1)
    outlier_best_label = le.inverse_transform(outlier_best_class)

    # Build recovery suggestions
    recovery_suggestions = {}
    for idx, (gi, pred_label, conf) in enumerate(zip(
        outlier_indices, outlier_best_label, outlier_best_conf
    )):
        recovery_suggestions[int(gi)] = {
            'suggested_cluster': int(pred_label),
            'confidence': float(conf),
            'tt_code': tt_codes[gi],
            'title': title_lookup.get(tt_codes[gi], tt_codes[gi])
        }

    # Summary stats
    high_conf = sum(1 for v in recovery_suggestions.values() if v['confidence'] >= RECOVERY_THRESHOLD)
    mid_conf = sum(1 for v in recovery_suggestions.values() if 0.3 <= v['confidence'] < RECOVERY_THRESHOLD)
    low_conf = sum(1 for v in recovery_suggestions.values() if v['confidence'] < 0.3)

    print(f'Outlier Recovery Results ({len(recovery_suggestions)} outliers analyzed)')
    print(f'{"="*60}')
    print(f'  ≥{RECOVERY_THRESHOLD:.0%} confidence (suggest recovery):  {high_conf}')
    print(f'  30-{RECOVERY_THRESHOLD:.0%} confidence (borderline):       {mid_conf}')
    print(f'  <30% confidence (true outliers):        {low_conf}')
    print(f'{"="*60}')

    # Show top recovery candidates
    top_recoveries = sorted(
        recovery_suggestions.values(),
        key=lambda x: x['confidence'], reverse=True
    )[:15]
    print(f'\nTop 15 recovery candidates:')
    print(f'{"Title":<45} {"Cluster":<10} {"Confidence"}')
    print("-" * 70)
    for r in top_recoveries:
        c_name = cluster_names.get(r['suggested_cluster'], {}).get('name', f'Cluster {r["suggested_cluster"]}') if 'cluster_names' in dir() else f'Cluster {r["suggested_cluster"]}'
        print(f'{r["title"][:44]:<45} {r["suggested_cluster"]:<10} {r["confidence"]:.1%}')
else:
    recovery_suggestions = {}
    print('No outliers to recover — all films are clustered')


### 8.5c-viz — Outlier Recovery Results

**Left panel (confidence histogram):** Shows the distribution of XGBoost's prediction confidence across all outlier films. The red dashed line marks the recovery threshold (50%). Films to the right of the line are strong recovery candidates; films clustering near 0% are true outliers with no clear home.

**Right panel (pie chart):** Breaks down the outliers into three tiers — recoverable (≥50%), borderline (30–50%), and true outliers (<30%). A healthy pipeline typically recovers 30–60% of outliers, with the remainder being genuinely uncategorisable films.

**What to look for:** If the histogram is heavily left-skewed (most outliers below 30%), the clusters may be too exclusive — consider lowering `min_cluster_size` to capture more films. If most outliers are above 50%, HDBSCAN may be too conservative — consider relaxing `cluster_selection_epsilon`.

**Inputs:** `recovery_suggestions` (dict), `RECOVERY_THRESHOLD`.

**Process:** Extract confidence values, plot histogram with threshold line and pie chart with three tiers.

**Outputs:** Matplotlib figure (displayed inline). No new variables created.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  OUTLIER RECOVERY VISUALISATION — confidence histogram + breakdown pie     ║
# ║  Inputs:  recovery_suggestions, RECOVERY_THRESHOLD                         ║
# ║  Outputs: Matplotlib 2-panel figure (inline display only)                  ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Visualisation: Outlier Recovery Results ──────────────────────────────────
if recovery_suggestions:
    _fig, _axes = plt.subplots(1, 2, figsize=(14, 5))

    # ── Left: Confidence distribution of outlier predictions ─────────────────
    _confs = [v['confidence'] for v in recovery_suggestions.values()]
    _axes[0].hist(_confs, bins=30, color='#FF9800', edgecolor='white', alpha=0.9)
    _axes[0].axvline(RECOVERY_THRESHOLD, color='#F44336', linestyle='--', linewidth=2,
                     label=f'Recovery threshold ({RECOVERY_THRESHOLD:.0%})')
    _axes[0].set_xlabel('XGBoost Prediction Confidence', fontsize=12)
    _axes[0].set_ylabel('Number of Outliers', fontsize=12)
    _axes[0].set_title('Outlier Recovery Confidence Distribution', fontsize=14, fontweight='bold', pad=12)
    _axes[0].legend(fontsize=11)
    _axes[0].spines[['top', 'right']].set_visible(False)

    # ── Right: Recovery breakdown ────────────────────────────────────────────
    _high = sum(1 for c in _confs if c >= RECOVERY_THRESHOLD)
    _mid = sum(1 for c in _confs if 0.3 <= c < RECOVERY_THRESHOLD)
    _low = sum(1 for c in _confs if c < 0.3)

    _labels = [f'Recoverable\n(≥{RECOVERY_THRESHOLD:.0%})', 'Borderline\n(30-50%)', 'True Outliers\n(<30%)']
    _sizes = [_high, _mid, _low]
    _pie_colors = ['#4CAF50', '#FF9800', '#F44336']
    _explode = (0.05, 0, 0)

    _wedges, _texts, _autotexts = _axes[1].pie(
        _sizes, labels=_labels, colors=_pie_colors, autopct='%1.0f%%',
        startangle=90, explode=_explode, textprops={'fontsize': 11}
    )
    for _at in _autotexts:
        _at.set_fontweight('bold')
        _at.set_fontsize(13)

    _axes[1].set_title(f'Outlier Breakdown ({len(_confs)} total)',
                       fontsize=14, fontweight='bold', pad=12)

    _fig.tight_layout()
    plt.show()

    print(f'\n  Recoverable: {_high} films with ≥{RECOVERY_THRESHOLD:.0%} confidence')
    print(f'  Borderline:  {_mid} films with 30-{RECOVERY_THRESHOLD:.0%} confidence')
    print(f'  True outliers: {_low} films with <30% confidence')
else:
    print('No outliers to visualise — all films are clustered')

## 9 — LLM Cluster Naming (SHAP-Enhanced)

### Why Use an LLM for Naming?

Each cluster at this point is just "Cluster 7" — a number with no meaning. We could name them manually by inspecting movie lists, but with 40+ clusters that's tedious and subjective. Instead, we prompt a local LLM with each cluster's **SHAP feature profile** and ask it to generate an evocative, descriptive name.

### How the Naming Works

For each cluster, the naming prompt includes:

1. **Top 10 SHAP features** ranked by importance — e.g., "revenge (0.0842), stylized violence (0.0731), crime (0.0654)..."
2. **Unique vs shared features** — Features that appear in multiple clusters are flagged so the LLM knows which traits are *unique* to this rail vs common across several.
3. **Representative movie titles** — The 5 highest-confidence members give the LLM concrete examples to anchor the name.

The LLM returns a JSON response with a **name** (3–6 words, like "Gritty Urban Crime Thrillers") and a **one-sentence description**.

### Why SHAP Features Instead of Raw Statistics?

Earlier approaches used centroid-delta features (comparing a cluster's mean feature values to the global mean). This misses *feature interactions* — if "heist" and "ensemble cast" *together* define a cluster but neither alone is unusual, centroid-delta would miss both. SHAP captures these interactions because it measures each feature's marginal contribution to the XGBoost prediction.

### Local LLM via Ollama

We use a local Ollama model (`llama3.2:3b`) rather than a cloud API for privacy and reproducibility. The model runs entirely on-device. Requires `ollama serve` to be running.

**Inputs:** `cluster_shap_features`, `cluster_labels`, `probabilities`, `tt_codes`, `title_lookup`, `OLLAMA_MODEL`, `OLLAMA_BASE_URL`.

**Process:**
1. This cell defines helper functions (`get_shap_features`, `build_all_cluster_top_features_shap`, `query_ollama`, `name_cluster_shap`).
2. The next cell executes naming for all clusters.

**Outputs:** Functions available in the namespace. The naming execution happens in the following cell.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  LLM NAMING FUNCTIONS — SHAP-enhanced cluster naming via Ollama            ║
# ║  Inputs:  cluster_shap_features, OLLAMA_MODEL, OLLAMA_BASE_URL            ║
# ║  Outputs: Functions: get_shap_features, build_all_cluster_top_features_shap║
# ║           query_ollama, name_cluster_shap, format_features_short           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import requests
import json as json_module

# ═══════════════════════════════════════════════════════════════════
# SHAP-based discriminative features (replaces centroid-delta)
# ═══════════════════════════════════════════════════════════════════

def get_shap_features(cluster_id, top_n=10):
    """Get top SHAP features for a cluster (pre-computed in Section 8.5b)."""
    return cluster_shap_features.get(cluster_id, [])[:top_n]


def build_all_cluster_top_features_shap(top_n=10):
    """Build dict of top SHAP feature names per cluster (for cross-cluster dedup)."""
    return {cid: [f[0] for f in feats[:top_n]]
            for cid, feats in cluster_shap_features.items()}


def format_features_short(feat_list, n=5):
    """Format feature list for printing."""
    return [f'{name} ({score:.4f})' for name, score in feat_list[:n]]


# ═══════════════════════════════════════════════════════════════════
# Ollama LLM interface
# ═══════════════════════════════════════════════════════════════════

def query_ollama(prompt, model=None):
    model = model or OLLAMA_MODEL
    try:
        response = requests.post(
            f'{OLLAMA_BASE_URL}/api/generate',
            json={'model': model, 'prompt': prompt, 'stream': False,
                  'options': {'temperature': 0.3}},
            timeout=180
        )
        data = response.json()
        if 'error' in data:
            raise RuntimeError(f"Ollama error: {data['error']}")
        return data['response'].strip()
    except Exception as e:
        print(f'  Ollama call failed: {e}')
        return f'{{"name": "Unnamed", "description": "LLM unavailable: {e}"}}'


# ═══════════════════════════════════════════════════════════════════
# SHAP-enhanced naming
# ═══════════════════════════════════════════════════════════════════

def name_cluster_shap(cluster_id, all_tops, sample_tt_codes=None, title_lookup=None):
    """Name a cluster using SHAP feature importances from XGBoost."""
    top_feats = get_shap_features(cluster_id, top_n=10)

    if not top_feats:
        return {'name': f'Cluster {cluster_id}', 'description': 'No SHAP data available',
                'top_features': [], 'method': 'fallback'}

    # Identify features shared with other clusters
    my_feat_names = [f[0] for f in top_feats]
    shared = set()
    for other_cid, other_feats in all_tops.items():
        if other_cid == cluster_id:
            continue
        shared |= set(my_feat_names) & set(other_feats)

    # Build prompt
    pos_str = '\n'.join([f'  - {name}: SHAP importance {score:.4f}' for name, score in top_feats])

    diff_str = ''
    if shared:
        shared_list = ', '.join(sorted(shared))
        unique_feats = [(n, s) for n, s in top_feats if n not in shared]
        if unique_feats:
            unique_str = '\n'.join([f'  - {n}: {s:.4f} (unique to this cluster)'
                                    for n, s in unique_feats[:5]])
            diff_str = (
                f'\n\nNOTE: Some features above (specifically: {shared_list}) also appear in other clusters.'
                f'\nThe following are UNIQUE to this cluster:\n{unique_str}'
            )

    sample_str = ''
    if sample_tt_codes and title_lookup:
        titles = [title_lookup.get(str(tt), str(tt)) for tt in sample_tt_codes[:5]]
        sample_str = f'\nRepresentative movies (highest confidence members): {", ".join(titles)}'

    prompt = (
        "You are a film taxonomy expert. A cluster of movies has been grouped by "
        "the similarity of their content profile. The features below are ranked by SHAP importance — "
        "a machine learning technique that measures how much each feature contributes to identifying "
        "films in THIS cluster vs all others. Higher values mean the feature is more important "
        "for distinguishing this cluster.\n\n"
        f"KEY DISTINGUISHING FEATURES (SHAP-ranked):\n{pos_str}{diff_str}\n"
        f"{sample_str}\n\n"
        "Based on this profile, give this cluster a precise, evocative name (3-6 words) "
        "and a one-sentence description of what unifies these films.\n\n"
        "Respond in JSON only:\n"
        '{"name": "cluster name", "description": "one sentence"}'
    )

    raw = query_ollama(prompt)
    try:
        result = json_module.loads(raw)
    except json_module.JSONDecodeError:
        result = {'name': f'Cluster {cluster_id}', 'description': raw}

    result['top_features'] = format_features_short(top_feats)
    result['method'] = 'SHAP (XGBoost)'
    if shared:
        result['shared_with_other_rails'] = list(shared)
    return result


print('SHAP-enhanced LLM naming functions defined')
print(f'  Model: {OLLAMA_MODEL}')
print(f'  Endpoint: {OLLAMA_BASE_URL}')
print(f'  XGBoost CV accuracy: {cv_scores.mean():.1%}')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  LLM NAMING EXECUTION — name all clusters using Ollama + SHAP features    ║
# ║  Inputs:  naming functions (Cell 49), cluster_labels, probabilities,      ║
# ║           tt_codes, title_lookup, OLLAMA_MODEL                             ║
# ║  Outputs: cluster_names (dict: cluster_id → {name, description, ...})     ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# Requires: ollama serve  (with the model already pulled)

cluster_names = {}

# ── Pre-compute top features for all clusters (for dedup) ────────────────────
all_cluster_tops = build_all_cluster_top_features_shap(top_n=10)

# ── Name top-level clusters using SHAP features ─────────────────────────────
print(f'Naming {n_clusters} clusters via Ollama ({OLLAMA_MODEL}) with SHAP features...\n')

for cluster_id in sorted(set(cluster_labels) - {-1}):
    mask       = cluster_labels == cluster_id
    _mask_idx  = np.where(mask)[0]
    _conf_order = np.argsort(probabilities[_mask_idx])[::-1]
    sample_tts = [tt_codes[_mask_idx[j]] for j in _conf_order[:5]]
    result     = name_cluster_shap(cluster_id, all_cluster_tops, sample_tts, title_lookup)
    cluster_names[cluster_id] = result

    avg_conf = probabilities[mask].mean()
    print(f'{"="*70}')
    print(f'Rail {cluster_id}: {result["name"]}  ({mask.sum()} films, {avg_conf:.0%} avg conf)')
    print(f'  {result["description"]}')
    print(f'  Top SHAP features: {", ".join(result["top_features"])}')
    if result.get('shared_with_other_rails'):
        print(f'  Shared with other rails: {", ".join(result["shared_with_other_rails"])}')
    sample_titles = [title_lookup.get(str(tt), str(tt)) for tt in sample_tts]
    print(f'  Top confidence examples: {", ".join(sample_titles)}')

print(f'\nNaming complete: {len(cluster_names)} rails (SHAP-enhanced)')


### 9.1 — Final Rails Overview

This is the payoff — the complete catalogue of named streaming rails, visualised as a bar chart sorted by size.

**Colour encoding:** Bar colour represents the average membership probability for that rail (using a Red → Yellow → Green gradient). Green bars have highly confident member assignments; red bars have more borderline members. Low-confidence rails might need parameter tuning or manual curation.

**What to look for:**
- **Meaningful names** — Do the rail names match the movies inside them? Cross-reference with the SHAP features and movie lists.
- **Balanced sizes** — Very large rails might benefit from sub-clustering (hierarchical splits). Very small rails might be too niche to be useful as a streaming category.
- **Confidence distribution** — Rails with low average confidence may have fuzzy boundaries and could benefit from further investigation.

**Inputs:** `cluster_names`, `cluster_labels`, `probabilities`.

**Process:** Build a sorted list of rails by size, plot horizontal bar chart with RdYlGn colormap, annotate with film count and confidence percentage.

**Outputs:** Matplotlib figure (displayed inline). No new variables created.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  RAILS OVERVIEW VISUALISATION — horizontal bar chart of all named rails    ║
# ║  Inputs:  cluster_names, cluster_labels, probabilities                     ║
# ║  Outputs: Matplotlib figure (inline display only)                          ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

# ── Visualisation: Named Rails Overview ───────────────────────────────────────
import matplotlib.cm as cm

_rail_data = []
for _cid, _info in cluster_names.items():
    _name = _info['name'] if isinstance(_info, dict) else str(_info)
    _count = int((cluster_labels == _cid).sum())
    _avg_conf = float(probabilities[cluster_labels == _cid].mean())
    _rail_data.append((_cid, _name, _count, _avg_conf))

# Sort by size descending
_rail_data.sort(key=lambda x: x[2], reverse=True)

_names = [f'{r[1]}' for r in _rail_data]
_counts = [r[2] for r in _rail_data]
_confs = [r[3] for r in _rail_data]

# Color by average confidence
_norm = plt.Normalize(vmin=min(_confs) - 0.05, vmax=max(_confs))
_cmap = cm.get_cmap('RdYlGn')
_bar_colors = [_cmap(_norm(c)) for c in _confs]

_fig, _ax = plt.subplots(figsize=(14, max(8, len(_names) * 0.38)))

_bars = _ax.barh(range(len(_names)), _counts, color=_bar_colors, edgecolor='white', linewidth=0.5)
_ax.set_yticks(range(len(_names)))
_ax.set_yticklabels(_names, fontsize=9)
_ax.set_xlabel('Number of Movies', fontsize=13)
_ax.set_title(f'Final Rails — {len(_names)} Named Clusters (color = avg confidence)',
              fontsize=15, fontweight='bold', pad=14)
_ax.invert_yaxis()
_ax.spines[['top', 'right']].set_visible(False)

# Add count + confidence labels
for _bar, _count, _conf in zip(_bars, _counts, _confs):
    _ax.text(_bar.get_width() + 3, _bar.get_y() + _bar.get_height()/2,
             f'{_count} films  ({_conf:.0%})', va='center', fontsize=9, color='#333')

# Colorbar
_sm = cm.ScalarMappable(cmap=_cmap, norm=_norm)
_sm.set_array([])
_cbar = _fig.colorbar(_sm, ax=_ax, shrink=0.6, label='Avg Membership Probability')

_fig.tight_layout()
plt.show()

print(f'\n  {len(_names)} rails  |  {sum(_counts):,} clustered films  |  {n_outliers} outliers')
print(f'  Largest rail: {_names[0]} ({_counts[0]} films)')
print(f'  Smallest rail: {_names[-1]} ({_counts[-1]} films)')

## 10 — Export Artifacts

Serialises **all** pipeline outputs into a single `pipeline_artifacts.pkl` file that the Streamlit dashboard loads at startup. This is the bridge between the notebook (exploration/tuning) and the dashboard (presentation/interaction).

The artifact dictionary includes every output the dashboard might need: embeddings for visualisation, cluster labels and probabilities for filtering, feature matrices for on-the-fly analysis, SHAP profiles for cluster detail pages, recovery suggestions for the outlier review panel, movie metadata for display, and the full pipeline config for reproducibility.

### Why One File?

A single serialised dict is simpler to manage than dozens of CSV exports. The dashboard calls `joblib.load()` once and has everything it needs. `joblib` with zlib compression keeps the file size manageable (~10–30 MB depending on cluster count).

**Inputs:** All pipeline outputs — embeddings, labels, probabilities, feature matrices, IDF weights, SHAP features, recovery suggestions, cluster names, metadata, and config.

**Process:** Bundle everything into a dict and save with `joblib.dump(compress=('zlib', 3))`.

**Outputs:** `Dashboard/results/pipeline_artifacts.pkl` — the single file the dashboard loads at startup.

In [48]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  EXPORT ARTIFACTS — serialise all pipeline outputs for the dashboard       ║
# ║  Inputs:  All pipeline variables (embeddings, labels, models, metadata)    ║
# ║  Outputs: Dashboard/results/pipeline_artifacts.pkl                         ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import joblib
from datetime import datetime

results_dir = Path(RESULTS_DIR)
results_dir.mkdir(exist_ok=True)

# ── Bundle all pipeline outputs into a single dict ────────────────────────────
artifacts = {
    # UMAP embeddings (three dimensionalities)
    'embedding_cluster': embedding_cluster,   # 20D — used for clustering
    'embedding_3d': embedding_3d,             # 3D  — used for scatter plots
    'embedding_2d': embedding_2d,             # 2D  — slice of 3D for flat plots

    # HDBSCAN outputs
    'cluster_labels': cluster_labels,         # Array of ints (-1 = outlier)
    'probabilities': clusterer.probabilities_,# Membership confidence (0-1)
    'outlier_scores': clusterer.outlier_scores_,  # GLOSH outlier scores
    'n_clusters': n_clusters,                 # Total cluster count

    # Feature matrices
    'X': X,                                   # Raw unweighted features
    'X_weighted': X_weighted,                 # IDF-weighted features (UMAP + XGBoost input)
    'feature_names': feature_names,           # Column names for X / X_weighted
    'tt_codes': tt_codes,                     # IMDb IDs aligned with matrix rows
    'idf_weights': idf_weights,               # 248 IDF weights (one per genome feature)

    # Feature type breakdowns
    'genome_cols': genome_cols,               # List of 248 genome feature names
    'genre_cols': genre_cols,                 # List of ~23 genre feature names
    'decade_cols': decade_cols,               # List of ~11 decade feature names

    # XGBoost / SHAP / Naming outputs
    'cluster_names': cluster_names,           # {cluster_id: {name, description, ...}}
    'cluster_shap_features': cluster_shap_features,  # {cluster_id: [(feat, shap), ...]}
    'recovery_suggestions': recovery_suggestions,    # {idx: {cluster, confidence, ...}}
    'xgb_cv_accuracy': float(cv_scores.mean()),      # Mean 5-fold CV accuracy
    'xgb_cv_std': float(cv_scores.std()),             # Std of 5-fold CV accuracy

    # Metadata
    'title_lookup': title_lookup,             # {imdb_id: title}
    'jordan_df': jordan_df,                   # Metadata DataFrame (year, rating, etc.)

    # Pipeline config (for reproducibility)
    'config': {
        'PIPELINE': 'idf_xgboost_experimental',
        'PREPROCESSING': 'IDF (TfidfTransformer) on genome features',
        'IDF_TRANSFORM': 'genome features only (binary cols untouched)',
        'UMAP_N_COMPONENTS': UMAP_N_COMPONENTS,
        'UMAP_N_NEIGHBORS': UMAP_N_NEIGHBORS,
        'UMAP_MIN_DIST': UMAP_MIN_DIST,
        'UMAP_METRIC': UMAP_METRIC,
        'HDBSCAN_MIN_CLUSTER_SIZE': HDBSCAN_MIN_CLUSTER_SIZE,
        'HDBSCAN_MIN_SAMPLES': HDBSCAN_MIN_SAMPLES,
        'HDBSCAN_EPSILON': HDBSCAN_EPSILON,
        'HDBSCAN_SELECTION_METHOD': HDBSCAN_SELECTION_METHOD,
        'INCLUDE_GENRE_FEATURES': INCLUDE_GENRE_FEATURES,
        'INCLUDE_DECADE_FEATURE': INCLUDE_DECADE_FEATURE,
        'OLLAMA_MODEL': OLLAMA_MODEL,
        'timestamp': datetime.now().isoformat(),
    }
}

# ── Save with compression ────────────────────────────────────────────────────
out_path = results_dir / 'pipeline_artifacts.pkl'
joblib.dump(artifacts, out_path, compress=('zlib', 3))  # zlib level 3 = good balance
print(f'Pipeline artifacts saved to {out_path}')
print(f'Keys: {list(artifacts.keys())}')

Pipeline artifacts saved to Dashboard/results/pipeline_artifacts.pkl
Keys: ['embedding_cluster', 'embedding_3d', 'embedding_2d', 'cluster_labels', 'probabilities', 'outlier_scores', 'n_clusters', 'X', 'X_weighted', 'feature_names', 'tt_codes', 'idf_weights', 'genome_cols', 'genre_cols', 'decade_cols', 'cluster_names', 'title_lookup', 'jordan_df', 'config']


## Summary

Prints a comprehensive pipeline report so you can verify the full run at a glance. This is the cell to check if something seems off — it shows the dataset size, preprocessing method, all hyperparameters, cluster count, quality metrics, validation accuracy, and outlier recovery stats in one place.

**Key numbers to watch:**
- **XGBoost CV accuracy** ≥ 90% → clusters are well-defined
- **Silhouette** ≥ 0.4 → good separation
- **DBCV** ≥ 0.4 → good density-based validity
- **Outlier %** ≤ 10% → most films are clustered
- **Recovery rate** — what fraction of outliers can be confidently reassigned

**Inputs:** All pipeline variables (aggregated).

**Outputs:** Formatted print summary (no new variables).

In [49]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  PIPELINE SUMMARY — complete run report                                    ║
# ║  Inputs:  All pipeline variables (aggregated)                              ║
# ║  Outputs: Formatted print summary (no new variables)                       ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

print('=' * 60)
print('PIPELINE SUMMARY — EXPERIMENTAL (IDF + XGBoost)')
print('=' * 60)
print(f'  Movies:          {X.shape[0]:,}')
print(f'  Features:        {X.shape[1]} ({len(genome_cols)} genome + {len(genre_cols)} genre + {len(decade_cols)} decade)')
print(f'  Preprocessing:   IDF (TfidfTransformer) on {len(genome_cols)} genome features')
print(f'  UMAP input:      {X_weighted.shape[1]} IDF-weighted features')
print(f'  XGBoost input:   {X_weighted.shape[1]} IDF-weighted features (original names)')
print(f'  UMAP:            {UMAP_N_COMPONENTS}D, {UMAP_N_NEIGHBORS} neighbors, min_dist={UMAP_MIN_DIST}, {UMAP_METRIC}')
print(f'  HDBSCAN:         min_size={HDBSCAN_MIN_CLUSTER_SIZE}, min_samples={HDBSCAN_MIN_SAMPLES}, eps={HDBSCAN_EPSILON}, {HDBSCAN_SELECTION_METHOD}')
print(f'  Clusters:        {n_clusters}')
print(f'  Outliers:        {n_outliers} ({100 * n_outliers / n_total:.1f}%)')
print(f'  Silhouette:      {sil:.3f}')
print(f'  DBCV:            {dbcv_score:.3f}')
print(f'  ---')
print(f'  XGBoost CV:      {cv_scores.mean():.1%} ± {cv_scores.std():.1%}')
print(f'  SHAP features:   {len(cluster_shap_features)} clusters profiled')
_n_recovered = sum(1 for v in recovery_suggestions.values() if v["confidence"] >= RECOVERY_THRESHOLD)
print(f'  Outlier recovery: {_n_recovered} / {len(recovery_suggestions)} above {RECOVERY_THRESHOLD:.0%} threshold')
print(f'  Naming method:   SHAP (XGBoost)')
print(f'  Cluster names:   {len(cluster_names)}')
print(f'  Ollama model:    {OLLAMA_MODEL}')
print('=' * 60)

PIPELINE SUMMARY — IDF
  Movies:          8,000
  Features:        283 (247 genome + 23 genre + 13 decade)
  Preprocessing:   Per-type (genomes as-is, genres binary, decades one-hot binary)
  Weighting:       IDF raw (sklearn TfidfTransformer) on 247 genome features
  UMAP:            50D, 50 neighbors, min_dist=0.0, correlation
  HDBSCAN:         min_size=60, min_samples=3, eps=0.1, eom
  Clusters:        43
  Outliers:        589 (7.4%)
  Silhouette:      0.609
  DBCV:            0.415
  Cluster names:   43
  Ollama model:    llama3.2:3b


## 12 — Launch Dashboard

Starts the Streamlit dashboard locally using the artifacts exported in the previous cell. The dashboard provides an interactive interface for exploring clusters, browsing individual films, reviewing outlier recovery candidates, and visualising the feature space.

The dashboard has 7 pages: Overview, Feature Space, Cluster Explorer, Film Explorer, SHAP Analysis, Sweep Results, and Outlier Review. Each page loads data from the single `pipeline_artifacts.pkl` file.

**Inputs:** `Dashboard/app.py`, `pipeline_artifacts.pkl` from Section 10.

**Process:** Launch Streamlit as a background subprocess, wait 3 seconds for it to initialise, then open the browser.

**Outputs:** Running Streamlit server at `http://localhost:8501`. Use `proc.terminate()` to shut down.

In [50]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  LAUNCH DASHBOARD — start Streamlit server and open browser                ║
# ║  Inputs:  Dashboard/app.py, pipeline_artifacts.pkl                         ║
# ║  Outputs: Running Streamlit server at http://localhost:8501                 ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import subprocess, webbrowser, time

proc = subprocess.Popen(
    ['streamlit', 'run', 'Dashboard/app.py',
     '--server.port', '8501',
     '--server.headless', 'true',
     '--browser.gatherUsageStats', 'false'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

time.sleep(3)
webbrowser.open('http://localhost:8501')
print(f'Dashboard launched at http://localhost:8501')
print(f'Process PID: {proc.pid}')
print(f'Stop this cell or run proc.terminate() to shut down.')

Dashboard launched at http://localhost:8501
Process PID: 75485
Stop this cell or run proc.terminate() to shut down.
